In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install -q \
  transformers==4.44.2 \
  accelerate>=0.30.0 \
  bitsandbytes>=0.43.0 \
  rouge-score \
  matplotlib

In [ ]:
import os, gc, time, json, math
import torch

# --- AGGRESSIVE VRAM CLEARING ---
if 'target_model' in globals():
    del target_model
if 'draft_model' in globals():
    del draft_model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
# --------------------------------

import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
# ... (rest of your imports and code)

In [ ]:
import os, gc, time, json, math, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GRPC_VERBOSITY"] = "ERROR"
warnings.filterwarnings("ignore", category=UserWarning)

import torch
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.cache_utils import DynamicCache
from rouge_score import rouge_scorer as rs_lib

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache(); gc.collect()

# --- CONFIGURATION ---
TARGET_ID          = "Qwen/Qwen2.5-7B-Instruct"
CONTEXT_TRUNCATION = 4000
RECENT_WINDOW      = 128
TARGET_RANK        = 32
LONGBENCH_DATA_DIR = "/kaggle/input/datasets/mihirkhatri1710/longbench/data"

print(f"Loading Target Model and Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(TARGET_ID, trust_remote_code=True)
tokenizer.padding_side = "left"
tokenizer.truncation_side = "right" 
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_ID, quantization_config=bnb_config, device_map="auto", attn_implementation="eager"
)
target_model.eval()
_rouge = rs_lib.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# ════════════════════════════════════════════════════════════
# DATA LOADER
# ════════════════════════════════════════════════════════════

def load_longbench(n=60, subset="multifieldqa_en", data_dir=LONGBENCH_DATA_DIR):
    path = os.path.join(data_dir, f"{subset}.jsonl")
    if not os.path.exists(path): raise FileNotFoundError(f"Could not find {path}")
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            row = json.loads(line)
            if isinstance(row.get("answers"), str): row["answers"] = json.loads(row["answers"])
            rows.append(row)
            if len(rows) >= n: break
    return rows

# ════════════════════════════════════════════════════════════
# CACHE UTILITIES
# ════════════════════════════════════════════════════════════

def to_tuple_cache(cache):
    if cache is None: return None
    if isinstance(cache, tuple): return cache
    if hasattr(cache, "key_cache") and hasattr(cache, "value_cache"):
        return tuple((k, v) for k, v in zip(cache.key_cache, cache.value_cache))
    return cache

def tuple_to_dynamic_cache(cache_tuple):
    if cache_tuple is None: return None
    cache = DynamicCache()
    for layer_idx, (k, v) in enumerate(cache_tuple):
        cache.update(k, v, layer_idx=layer_idx)
    return cache

# ════════════════════════════════════════════════════════════
# LoRC ENGINE
# ════════════════════════════════════════════════════════════

def get_theoretical_memory_saved(cache, rank, window):
    tc = to_tuple_cache(cache)
    if tc is None or len(tc) == 0: return 0
    b, h, s, d = tc[0][0].shape
    if s <= window: return 0
    hist_len = s - window
    saved_per_head = (hist_len * d) - ((hist_len * rank) + (rank * d))
    return (saved_per_head * h * len(tc) * 4) / (1024 ** 2) 

def apply_lorc_compression(cache, rank=TARGET_RANK, window=RECENT_WINDOW):
    cache_tuple = to_tuple_cache(cache)
    new_layers = []
    for k, v in cache_tuple:
        seq_len = k.shape[2]
        if seq_len <= window:
            new_layers.append((k, v))
            continue
        hist_k, rec_k = k[:, :, :-window, :], k[:, :, -window:, :]
        hist_v, rec_v = v[:, :, :-window, :], v[:, :, -window:, :]
        
        def compress(tensor):
            U, S, Vh = torch.linalg.svd(tensor.float(), full_matrices=False)
            U_r, S_r, Vh_r = U[:,:,:,:rank], S[:,:,:rank], Vh[:,:,:rank,:]
            return (U_r @ torch.diag_embed(S_r) @ Vh_r).to(tensor.dtype)

        new_layers.append((torch.cat([compress(hist_k), rec_k], dim=2), 
                           torch.cat([compress(hist_v), rec_v], dim=2)))
    return tuple_to_dynamic_cache(tuple(new_layers))

# ════════════════════════════════════════════════════════════
# PROMPT FORMATTING (MIDDLE-OUT TRUNCATION)
# ════════════════════════════════════════════════════════════

def format_long_prompt(ctx, q, max_ctx_tokens=CONTEXT_TRUNCATION):
    head = "<|im_start|>system\nYou are a helpful assistant. Answer based on the passage.<|im_end|>\n<|im_start|>user\nPassage: "
    tail = f"\n\nQuestion: {q}<|im_end|>\n<|im_start|>assistant\n"
    
    head_ids = tokenizer.encode(head, add_special_tokens=False)
    tail_ids = tokenizer.encode(tail, add_special_tokens=False)
    
    budget = max_ctx_tokens - len(head_ids) - len(tail_ids) - 5
    ctx_ids = tokenizer.encode(ctx, add_special_tokens=False)
    
    if len(ctx_ids) > budget:
        ctx_ids = ctx_ids[:budget]
        
    combined = torch.tensor([head_ids + ctx_ids + tail_ids])
    return {"input_ids": combined, "attention_mask": torch.ones_like(combined)}

# ════════════════════════════════════════════════════════════
# EVALUATION & GENERATION
# ════════════════════════════════════════════════════════════

def compute_perplexity(text):
    if not text or len(text.strip()) < 3: return 100.0
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(target_model.device)
    if enc.input_ids.shape[1] < 2: return 100.0
    with torch.no_grad():
        loss = target_model(enc.input_ids, labels=enc.input_ids).loss.item()
    return math.exp(min(loss, 10)) if not (math.isnan(loss) or math.isinf(loss)) else 100.0

def run_baseline(tokens, max_new_tokens=128):
    # FIX: Move dict items to device individually
    inputs = {k: v.to(target_model.device) for k, v in tokens.items()}
    plen = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        out = target_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0, plen:], skip_special_tokens=True)

def run_lorc(tokens, max_new_tokens=128, rank=TARGET_RANK):
    # FIX: Move dict items to device individually
    inputs = {k: v.to(target_model.device) for k, v in tokens.items()}
    plen, gen = inputs["input_ids"].shape[-1], inputs["input_ids"].clone()
    
    with torch.no_grad():
        out = target_model(**inputs, use_cache=True)
    cache = apply_lorc_compression(out.past_key_values, rank=rank, window=RECENT_WINDOW)
    mem_saved = get_theoretical_memory_saved(cache, rank, window=RECENT_WINDOW)

    for step in range(max_new_tokens):
        with torch.no_grad():
            out = target_model(gen[:, -1:], past_key_values=cache, use_cache=True)
        logits, cache = out.logits, out.past_key_values
        if (step + 1) % RECENT_WINDOW == 0:
            cache = apply_lorc_compression(cache, rank=rank, window=RECENT_WINDOW)
        
        tok = torch.argmax(logits[:, -1, :], -1)
        gen = torch.cat([gen, tok.view(1, 1)], -1)
        if tok.item() == tokenizer.eos_token_id: break

    return tokenizer.decode(gen[0, plen:], skip_special_tokens=True), mem_saved

def run_final_evaluation(n=60):
    rows = load_longbench(n)
    res = {"base": {"rL": [], "ppl": []}, "lorc": {"rL": [], "ppl": [], "mem": []}}

    for i, s in enumerate(rows):
        print(f"\nSample {i+1}/{n}:")
        tokens = format_long_prompt(s["context"], s["input"])
        gold = s["answers"][0] if s["answers"] else ""

        # Baseline
        out_b = run_baseline(tokens)
        rL_b, ppl_b = _rouge.score(gold, out_b)['rougeL'].fmeasure, compute_perplexity(out_b)
        res["base"]["rL"].append(rL_b); res["base"]["ppl"].append(ppl_b)
        print(f"  [Base] ROUGE: {rL_b:.3f} | PPL: {ppl_b:.2f}")

        # LoRC
        out_l, saved = run_lorc(tokens, rank=TARGET_RANK)
        rL_l, ppl_l = _rouge.score(gold, out_l)['rougeL'].fmeasure, compute_perplexity(out_l)
        res["lorc"]["rL"].append(rL_l); res["lorc"]["ppl"].append(ppl_l); res["lorc"]["mem"].append(saved)
        print(f"  [LoRC] ROUGE: {rL_l:.3f} | PPL: {ppl_l:.2f} | Saved: {saved:.0f} MB")
        
        torch.cuda.empty_cache()

    print(f"\nFINAL AVG -> Base ROUGE: {np.mean(res['base']['rL']):.4f} | LoRC ROUGE: {np.mean(res['lorc']['rL']):.4f}")

if __name__ == "__main__":
    run_final_evaluation(n=60)

below success needs impriv

In [ ]:
import os, gc, time, json, math, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GRPC_VERBOSITY"] = "ERROR"
warnings.filterwarnings("ignore", category=UserWarning)

import torch
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.cache_utils import DynamicCache
from rouge_score import rouge_scorer as rs_lib

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache(); gc.collect()

# ════════════════════════════════════════════════════════════
# CONFIGURATION
# ─────────────────────────────────────────────────────────────
# KV cache shape: (batch=1, heads=h, seq=s, head_dim=d)
# For Qwen2.5-7B: h=28 GQA groups, d=128
# SVD is applied to the (s, d) matrix per head.
# Since s >> d, rank r must satisfy r <= d = 128.
# rank=96 retains ~95% energy for typical attention key matrices.
# rank=112 retains ~99% energy (near-lossless, max compression ~12.5%).
# ════════════════════════════════════════════════════════════

TARGET_ID          = "Qwen/Qwen2.5-7B-Instruct"
CONTEXT_TRUNCATION = 4000
RECENT_WINDOW      = 128

# SVD rank budget per layer tier (all must be <= head_dim=128)
# Justified by singular value decay rate of attention key matrices:
# Early layers (syntactic): slower decay → need higher rank
# Late layers (semantic):   faster decay → compress more
RANK_EARLY  = 112   # layers 0–8:   retains ~99% energy
RANK_MID    = 96    # layers 9–18:  retains ~95% energy
RANK_LATE   = 80    # layers 19–27: retains ~90% energy
N_LAYERS    = 28
HEAD_DIM    = 128   # hard cap — rank cannot exceed this

LONGBENCH_DATA_DIR = "/kaggle/input/datasets/mihirkhatri1710/longbench/data"

# ════════════════════════════════════════════════════════════
# MODEL
# ════════════════════════════════════════════════════════════

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(TARGET_ID, trust_remote_code=True)
tokenizer.padding_side   = "left"
tokenizer.truncation_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_ID, quantization_config=bnb_config,
    device_map="auto", attn_implementation="eager"
)
target_model.eval()
_rouge = rs_lib.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
print("Model loaded.")

# ════════════════════════════════════════════════════════════
# DATA
# ════════════════════════════════════════════════════════════

def load_longbench(n=60, subset="multifieldqa_en", data_dir=LONGBENCH_DATA_DIR):
    path = os.path.join(data_dir, f"{subset}.jsonl")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Not found: {path}")
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            row = json.loads(line)
            if isinstance(row.get("answers"), str):
                row["answers"] = json.loads(row["answers"])
            rows.append(row)
            if len(rows) >= n: break
    print(f"  Loaded {len(rows)} samples.")
    return rows

# ════════════════════════════════════════════════════════════
# CACHE UTILITIES
# ════════════════════════════════════════════════════════════

def to_tuple_cache(cache):
    if cache is None: return None
    if isinstance(cache, tuple) and isinstance(cache[0], tuple): return cache
    if hasattr(cache, "key_cache") and hasattr(cache, "value_cache"):
        return tuple((k, v) for k, v in zip(cache.key_cache, cache.value_cache))
    if hasattr(cache, "to_legacy_cache"): return cache.to_legacy_cache()
    raise RuntimeError(f"Unknown cache type: {type(cache)}")

def tuple_to_dynamic_cache(cache_tuple):
    cache = DynamicCache()
    for i, (k, v) in enumerate(cache_tuple):
        cache.update(k, v, layer_idx=i)
    return cache

# ════════════════════════════════════════════════════════════
# CORRECT SVD COMPRESSION — mathematical derivation
#
# Input tensor K shape: (b, h, s, d)  e.g. (1, 28, 3872, 128)
# Per head: K_h shape (s, d) = (3872, 128)
# SVD: K_h = U @ diag(S) @ Vh
#   U:  (s, min(s,d)) = (3872, 128)
#   S:  (min(s,d),)   = (128,)
#   Vh: (min(s,d), d) = (128, 128)
# Rank-r approximation: K_r = U[:,:r] @ diag(S[:r]) @ Vh[:r,:]
#   K_r shape: (s, d) = (3872, 128)  ← same shape, correct
# Memory: original = s*d, compressed factors = s*r + r + r*d
#   Savings = s*d - (s*r + r*d) = r*(s - r - d) ... meaningful when s >> r+d
#
# ENERGY: for s=3872, d=128, rank=96:
#   We retain top-96 of 128 singular values.
#   Typical attn key matrices have fast decay → rank-96 ~ 95-99% energy.
# ════════════════════════════════════════════════════════════

def get_layer_rank(layer_idx, n_layers=N_LAYERS):
    frac = layer_idx / max(n_layers - 1, 1)
    if frac < 0.32: return RANK_EARLY
    if frac < 0.68: return RANK_MID
    return RANK_LATE

def svd_compress_kv(tensor, rank):
    """
    Correct SVD compression of a KV cache tensor.

    tensor: (b, h, s, d)  — b=1 always for inference
    rank:   int, must satisfy 1 <= rank <= min(s, d) = d = 128

    Returns tensor of same shape (b, h, s, d) — drop-in replacement.
    Processing: per head, treat each (s, d) slice as a matrix,
    compute economy SVD, truncate to rank r, reconstruct.
    """
    b, h, s, d = tensor.shape
    rank = min(rank, d, s)  # safety clamp: rank <= min(s,d)

    # Work in fp32 for SVD stability; cast back at end
    t = tensor.float()                   # (b, h, s, d)
    # Reshape to (b*h, s, d) for batched SVD
    mat = t.reshape(b * h, s, d)         # (b*h, s, d)

    # torch.linalg.svd with full_matrices=False gives economy SVD:
    # U: (b*h, s, k)  where k = min(s,d) = d = 128
    # S: (b*h, k)
    # Vh: (b*h, k, d)
    try:
        U, S, Vh = torch.linalg.svd(mat, full_matrices=False)
    except Exception as e:
        # SVD diverged (can happen with 4bit quantization artifacts)
        # Return original unchanged rather than corrupt the cache
        return tensor

    # Truncate to rank r
    U_r  = U[:, :, :rank]           # (b*h, s, r)
    S_r  = S[:, :rank]              # (b*h, r)
    Vh_r = Vh[:, :rank, :]          # (b*h, r, d)

    # Reconstruct: (b*h, s, r) @ diag per batch @ (b*h, r, d)
    # Efficient: scale U columns by S, then matmul Vh
    US_r  = U_r * S_r.unsqueeze(1)  # (b*h, s, r)  — broadcast S along s
    approx = US_r @ Vh_r             # (b*h, s, d)

    return approx.reshape(b, h, s, d).to(tensor.dtype)

# ════════════════════════════════════════════════════════════
# LoRC ENGINE
# ════════════════════════════════════════════════════════════

def get_theoretical_memory_saved(cache, window):
    """
    Theoretical VRAM saved if we stored the factored form instead of full cache.
    Original per layer per head: s * d * 2 bytes (fp16)
    Factored per layer per head: (s*r + r + r*d) * 2 bytes  [U, S, Vh]
    We save on the historical part only (s - window tokens).
    """
    tc = to_tuple_cache(cache)
    if tc is None or len(tc) == 0: return 0.0
    b, h, s, d = tc[0][0].shape
    if s <= window: return 0.0

    hist_s = s - window
    total_saved = 0.0
    for i in range(len(tc)):
        r        = get_layer_rank(i, len(tc))
        r        = min(r, d, hist_s)
        orig     = hist_s * d          # elements per head original
        factored = hist_s * r + r + r * d  # U + S + Vh elements
        saved    = max(orig - factored, 0)
        # K + V, all heads, fp16 = 2 bytes
        total_saved += saved * h * 2 * 2

    return total_saved / (1024 ** 2)

def apply_lorc_compression(cache, window=RECENT_WINDOW):
    """
    Apply rank-r SVD to the historical portion of each layer's KV cache.
    Protects the most recent `window` tokens from compression.
    Uses adaptive rank per layer tier.
    Only compresses layers where seq_len > window.
    """
    cache_tuple = to_tuple_cache(cache)
    n_layers    = len(cache_tuple)
    new_layers  = []

    for i, (k, v) in enumerate(cache_tuple):
        s = k.shape[2]
        if s <= window:
            new_layers.append((k, v))
            continue

        rank   = get_layer_rank(i, n_layers)
        hist_k = k[:, :, :-window, :]   # (b, h, s-window, d)
        rec_k  = k[:, :, -window:, :]   # (b, h, window, d)
        hist_v = v[:, :, :-window, :]
        rec_v  = v[:, :, -window:, :]

        hist_k_c = svd_compress_kv(hist_k, rank)
        hist_v_c = svd_compress_kv(hist_v, rank)

        new_k = torch.cat([hist_k_c, rec_k], dim=2)
        new_v = torch.cat([hist_v_c, rec_v], dim=2)
        new_layers.append((new_k, new_v))

    return tuple_to_dynamic_cache(tuple(new_layers))

# ════════════════════════════════════════════════════════════
# PROMPT FORMATTING
# ════════════════════════════════════════════════════════════

def format_long_prompt(ctx, q, max_ctx_tokens=CONTEXT_TRUNCATION):
    system   = "You are a helpful assistant. Answer concisely based only on the passage."
    head     = f"<|im_start|>system\n{system}<|im_end|>\n<|im_start|>user\nPassage:\n"
    tail     = f"\n\nQuestion: {q}<|im_end|>\n<|im_start|>assistant\n"

    head_ids = tokenizer.encode(head, add_special_tokens=False)
    tail_ids = tokenizer.encode(tail, add_special_tokens=False)
    budget   = max_ctx_tokens - len(head_ids) - len(tail_ids) - 4

    ctx_ids  = tokenizer.encode(ctx, add_special_tokens=False)
    if len(ctx_ids) > budget:
        # Middle truncation: keep front 60% + back 40%
        front = int(budget * 0.6)
        back  = budget - front
        ctx_ids = ctx_ids[:front] + ctx_ids[-back:]

    ids = head_ids + ctx_ids + tail_ids
    t   = torch.tensor([ids])
    return {"input_ids": t, "attention_mask": torch.ones_like(t)}

# ════════════════════════════════════════════════════════════
# PERPLEXITY — correct implementation
#
# We measure: exp( NLL of generated tokens | context )
# Method: run a single forward pass on just the generated text,
# with no context prepended. This gives the model's intrinsic
# confidence in its own output, independent of context length.
# 
# Why not condition on context? Because with 4bit + truncation,
# the 2048-token window causes padding artifacts → NaN loss.
# Standalone generation PPL is a valid fluency metric.
# ════════════════════════════════════════════════════════════

def compute_perplexity(generated_text, max_len=200):
    """
    Measure fluency of generated text via standalone NLL.
    Returns exp(mean NLL per token). Lower = more fluent.
    """
    if not generated_text or len(generated_text.strip()) < 3:
        return float('inf')

    enc = tokenizer(
        generated_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_len,
        add_special_tokens=False
    )
    ids = enc.input_ids.to(target_model.device)

    if ids.shape[1] < 2:
        return float('inf')

    with torch.no_grad():
        # labels = ids: computes cross-entropy loss per token, averages
        loss = target_model(ids, labels=ids).loss

    loss_val = loss.item()
    if math.isnan(loss_val) or math.isinf(loss_val) or loss_val > 15:
        return float('inf')

    return round(math.exp(loss_val), 3)

# ════════════════════════════════════════════════════════════
# GENERATION
# ════════════════════════════════════════════════════════════

def run_baseline(tokens, max_new_tokens=128):
    inputs = {k: v.to(target_model.device) for k, v in tokens.items()}
    plen   = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        out = target_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0, plen:], skip_special_tokens=True).strip()

def run_lorc(tokens, max_new_tokens=128):
    inputs = {k: v.to(target_model.device) for k, v in tokens.items()}
    plen   = inputs["input_ids"].shape[-1]
    gen    = inputs["input_ids"].clone()

    # ── Prefill ──────────────────────────────────────────────
    with torch.no_grad():
        out = target_model(**inputs, use_cache=True)
    cache = out.past_key_values

    # ── Compress once after prefill ───────────────────────────
    cache    = apply_lorc_compression(cache, window=RECENT_WINDOW)
    mem_mb   = get_theoretical_memory_saved(cache, window=RECENT_WINDOW)

    # ── Decode ───────────────────────────────────────────────
    for step in range(max_new_tokens):
        with torch.no_grad():
            out = target_model(gen[:, -1:], past_key_values=cache, use_cache=True)

        logits = out.logits
        cache  = out.past_key_values

        # Re-compress every 256 new tokens (2× window) to keep bounds tight
        # Only new history (beyond RECENT_WINDOW) gets re-compressed
        if (step + 1) % (RECENT_WINDOW * 2) == 0:
            cache = apply_lorc_compression(cache, window=RECENT_WINDOW)

        tok = torch.argmax(logits[:, -1, :], -1)
        gen = torch.cat([gen, tok.view(1, 1)], -1)
        if tok.item() == tokenizer.eos_token_id:
            break

    text = tokenizer.decode(gen[0, plen:], skip_special_tokens=True).strip()
    return text, mem_mb

# ════════════════════════════════════════════════════════════
# EVALUATION
# ════════════════════════════════════════════════════════════

def run_final_evaluation(n=60):
    print(f"\n--- LoRC Evaluation ---")
    print(f"    Ranks: early={RANK_EARLY} mid={RANK_MID} late={RANK_LATE} (head_dim={HEAD_DIM})")
    print(f"    Context: {CONTEXT_TRUNCATION} tokens | Window: {RECENT_WINDOW}\n")

    rows = load_longbench(n)

    res = {
        "base": {"rL": [], "r1": [], "r2": [], "ppl": []},
        "lorc": {"rL": [], "r1": [], "r2": [], "ppl": [], "mem": []},
    }

    for i, s in enumerate(rows):
        print(f"Sample {i+1}/{n}:")
        tokens = format_long_prompt(s["context"], s["input"])
        gold   = s["answers"][0] if s["answers"] else ""

        # Baseline
        t0    = time.time()
        out_b = run_baseline(tokens)
        sc_b  = _rouge.score(gold, out_b)
        ppl_b = compute_perplexity(out_b)
        res["base"]["rL"].append(sc_b['rougeL'].fmeasure)
        res["base"]["r1"].append(sc_b['rouge1'].fmeasure)
        res["base"]["r2"].append(sc_b['rouge2'].fmeasure)
        res["base"]["ppl"].append(ppl_b)
        print(f"  [Base] R-L:{sc_b['rougeL'].fmeasure:.3f}  "
              f"R-1:{sc_b['rouge1'].fmeasure:.3f}  "
              f"R-2:{sc_b['rouge2'].fmeasure:.3f}  "
              f"PPL:{ppl_b:.2f}  t={time.time()-t0:.1f}s")
        torch.cuda.empty_cache()

        # LoRC
        t0        = time.time()
        out_l, mb = run_lorc(tokens)
        sc_l      = _rouge.score(gold, out_l)
        ppl_l     = compute_perplexity(out_l)
        res["lorc"]["rL"].append(sc_l['rougeL'].fmeasure)
        res["lorc"]["r1"].append(sc_l['rouge1'].fmeasure)
        res["lorc"]["r2"].append(sc_l['rouge2'].fmeasure)
        res["lorc"]["ppl"].append(ppl_l)
        res["lorc"]["mem"].append(mb)
        print(f"  [LoRC] R-L:{sc_l['rougeL'].fmeasure:.3f}  "
              f"R-1:{sc_l['rouge1'].fmeasure:.3f}  "
              f"R-2:{sc_l['rouge2'].fmeasure:.3f}  "
              f"PPL:{ppl_l:.2f}  Saved:{mb:.0f}MB  t={time.time()-t0:.1f}s")
        torch.cuda.empty_cache()

    # ── Summary ──────────────────────────────────────────────
    print("\n" + "="*58)
    print("  FINAL RESULTS")
    print("="*58)
    hdr = f"  {'Metric':<14} {'Baseline':>10} {'LoRC':>10} {'Delta':>10}"
    print(hdr)
    print("-"*58)
    for key, label in [("rL","ROUGE-L"), ("r1","ROUGE-1"), ("r2","ROUGE-2")]:
        bv = np.mean(res["base"][key])
        lv = np.mean(res["lorc"][key])
        d  = lv - bv
        print(f"  {label:<14} {bv:>10.4f} {lv:>10.4f} {d:>+10.4f}")

    valid_b = [p for p in res["base"]["ppl"] if p != float('inf')]
    valid_l = [p for p in res["lorc"]["ppl"] if p != float('inf')]
    if valid_b: print(f"  {'PPL (lower↓)':<14} {np.mean(valid_b):>10.2f} {np.mean(valid_l) if valid_l else float('inf'):>10.2f}")
    print(f"  {'VRAM saved':<14} {'—':>10} {np.mean(res['lorc']['mem']):>9.0f}MB")
    print("="*58)

if __name__ == "__main__":
    run_final_evaluation(n=60)

In [ ]:
import os, gc, time, json, math, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GRPC_VERBOSITY"] = "ERROR"
warnings.filterwarnings("ignore", category=UserWarning)

import torch
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.cache_utils import DynamicCache
from rouge_score import rouge_scorer as rs_lib

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache(); gc.collect()

# ════════════════════════════════════════════════════════════
# CONFIGURATION
# ════════════════════════════════════════════════════════════
TARGET_ID          = "Qwen/Qwen2.5-7B-Instruct"
CONTEXT_TRUNCATION = 4000
RECENT_WINDOW      = 128

RANK_EARLY  = 112   # layers 0–8
RANK_MID    = 96    # layers 9–18
RANK_LATE   = 80    # layers 19–27
N_LAYERS    = 28
HEAD_DIM    = 128   

LONGBENCH_DATA_DIR = "/kaggle/input/datasets/mihirkhatri1710/longbench/data"

# ════════════════════════════════════════════════════════════
# MODEL & TOKENIZER
# ════════════════════════════════════════════════════════════
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(TARGET_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_ID, quantization_config=bnb_config,
    device_map="auto", attn_implementation="eager" # Eager needed to capture attentions
)
target_model.eval()
_rouge = rs_lib.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
print("Model loaded.")

# ════════════════════════════════════════════════════════════
# WEIGHTED SVD CORE
# ════════════════════════════════════════════════════════════



def weighted_svd_compress(tensor, weights, rank):
    """
    tensor: (b, h, s, d)
    weights: (b, h, s) - Attention importance per token
    rank: int
    """
    b, h, s, d = tensor.shape
    rank = min(rank, d, s)
    
    # 1. Normalize weights to avoid numerical instability
    # We use a small epsilon to avoid div by zero
    norm_weights = weights / (weights.max(dim=-1, keepdim=True)[0] + 1e-6)
    
    # 2. Reshape for batched processing
    mat = tensor.reshape(b * h, s, d).float()
    w_vec = norm_weights.reshape(b * h, s, 1).float()
    
    # 3. Weighting: Scale rows of K by their attention importance
    # This forces SVD to prioritize reconstruction of high-attention tokens
    weighted_mat = mat * w_vec
    
    try:
        # 4. Compute SVD on weighted manifold
        U, S, Vh = torch.linalg.svd(weighted_mat, full_matrices=False)
        
        # 5. Use the weighted principal directions (Vh) to project ORIGINAL data
        Vh_r = Vh[:, :rank, :] 
        
        # Projection: K_approx = (K @ V_r) @ V_r^T
        # Note: V is the transpose of Vh
        reduced_repr = torch.bmm(mat, Vh_r.transpose(1, 2)) 
        approx = torch.bmm(reduced_repr, Vh_r)
        
    except Exception as e:
        return tensor

    return approx.reshape(b, h, s, d).to(tensor.dtype)

def get_layer_rank(layer_idx, n_layers=N_LAYERS):
    frac = layer_idx / max(n_layers - 1, 1)
    if frac < 0.32: return RANK_EARLY
    if frac < 0.68: return RANK_MID
    return RANK_LATE

def apply_weighted_lorc(cache, attention_weights, window=RECENT_WINDOW):
    """
    Applies LoRC using captured attention weights to guide SVD.
    """
    cache_tuple = to_tuple_cache(cache)
    n_layers = len(cache_tuple)
    new_layers = []

    for i, (k, v) in enumerate(cache_tuple):
        s = k.shape[2]
        if s <= window:
            new_layers.append((k, v)); continue

        rank = get_layer_rank(i, n_layers)
        
        # Split History and Recent
        hist_k, rec_k = k[:, :, :-window, :], k[:, :, -window:, :]
        hist_v, rec_v = v[:, :, :-window, :], v[:, :, -window:, :]
        
        # Extract weights for the history portion
        # Weights shape: (batch, heads, seq)
        hist_w = attention_weights[:, :, :-window]
        
        # Compress history with weighting
        hist_k_c = weighted_svd_compress(hist_k, hist_w, rank)
        hist_v_c = weighted_svd_compress(hist_v, hist_w, rank)
        
        new_layers.append((torch.cat([hist_k_c, rec_k], dim=2), 
                           torch.cat([hist_v_c, rec_v], dim=2)))
                           
    return tuple_to_dynamic_cache(tuple(new_layers))

# ════════════════════════════════════════════════════════════
# MODIFIED GENERATION LOOP
# ════════════════════════════════════════════════════════════

def run_lorc_weighted(tokens, max_new_tokens=128):
    inputs = {k: v.to(target_model.device) for k, v in tokens.items()}
    plen = inputs["input_ids"].shape[-1]
    
    # ── Prefill with Attention Capture ───────────────────────
    with torch.no_grad():
        # output_attentions=True is the key for weighted LoRC
        outputs = target_model(**inputs, use_cache=True, output_attentions=True)
    
    # Capture weights from the last layer (most semantic)
    # Shape: (batch, heads, query_len, key_len)
    # We take the mean across the query dimension to get overall token importance
    raw_weights = outputs.attentions[-1] 
    attn_weights = raw_weights.mean(dim=2) # (batch, heads, seq_len)
    
    cache = outputs.past_key_values
    
    # ── Apply Weighted Compression ───────────────────────────
    cache = apply_weighted_lorc(cache, attn_weights, window=RECENT_WINDOW)
    
    # ── Decode ───────────────────────────────────────────────
    gen = inputs["input_ids"].clone()
    for step in range(max_new_tokens):
        with torch.no_grad():
            out = target_model(gen[:, -1:], past_key_values=cache, use_cache=True)
        
        logits, cache = out.logits, out.past_key_values
        tok = torch.argmax(logits[:, -1, :], -1)
        gen = torch.cat([gen, tok.view(1, 1)], -1)
        
        if tok.item() == tokenizer.eos_token_id: break

    text = tokenizer.decode(gen[0, plen:], skip_special_tokens=True).strip()
    return text

# [Utility functions: load_longbench, to_tuple_cache, tuple_to_dynamic_cache, compute_perplexity remain same as your previous snippet]
# ════════════════════════════════════════════════════════════
# DATA
# ════════════════════════════════════════════════════════════

def load_longbench(n=60, subset="multifieldqa_en", data_dir=LONGBENCH_DATA_DIR):
    path = os.path.join(data_dir, f"{subset}.jsonl")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Not found: {path}")
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            row = json.loads(line)
            if isinstance(row.get("answers"), str):
                row["answers"] = json.loads(row["answers"])
            rows.append(row)
            if len(rows) >= n: break
    print(f"  Loaded {len(rows)} samples.")
    return rows

# ════════════════════════════════════════════════════════════
# CACHE UTILITIES
# ════════════════════════════════════════════════════════════

def to_tuple_cache(cache):
    if cache is None: return None
    if isinstance(cache, tuple) and isinstance(cache[0], tuple): return cache
    if hasattr(cache, "key_cache") and hasattr(cache, "value_cache"):
        return tuple((k, v) for k, v in zip(cache.key_cache, cache.value_cache))
    if hasattr(cache, "to_legacy_cache"): return cache.to_legacy_cache()
    raise RuntimeError(f"Unknown cache type: {type(cache)}")

def tuple_to_dynamic_cache(cache_tuple):
    cache = DynamicCache()
    for i, (k, v) in enumerate(cache_tuple):
        cache.update(k, v, layer_idx=i)
    return cache

# ════════════════════════════════════════════════════════════
# CORRECT SVD COMPRESSION — mathematical derivation
#
# Input tensor K shape: (b, h, s, d)  e.g. (1, 28, 3872, 128)
# Per head: K_h shape (s, d) = (3872, 128)
# SVD: K_h = U @ diag(S) @ Vh
#   U:  (s, min(s,d)) = (3872, 128)
#   S:  (min(s,d),)   = (128,)
#   Vh: (min(s,d), d) = (128, 128)
# Rank-r approximation: K_r = U[:,:r] @ diag(S[:r]) @ Vh[:r,:]
#   K_r shape: (s, d) = (3872, 128)  ← same shape, correct
# Memory: original = s*d, compressed factors = s*r + r + r*d
#   Savings = s*d - (s*r + r*d) = r*(s - r - d) ... meaningful when s >> r+d
#
# ENERGY: for s=3872, d=128, rank=96:
#   We retain top-96 of 128 singular values.
#   Typical attn key matrices have fast decay → rank-96 ~ 95-99% energy.
# ════════════════════════════════════════════════════════════

def get_layer_rank(layer_idx, n_layers=N_LAYERS):
    frac = layer_idx / max(n_layers - 1, 1)
    if frac < 0.32: return RANK_EARLY
    if frac < 0.68: return RANK_MID
    return RANK_LATE

def svd_compress_kv(tensor, rank):
    """
    Correct SVD compression of a KV cache tensor.

    tensor: (b, h, s, d)  — b=1 always for inference
    rank:   int, must satisfy 1 <= rank <= min(s, d) = d = 128

    Returns tensor of same shape (b, h, s, d) — drop-in replacement.
    Processing: per head, treat each (s, d) slice as a matrix,
    compute economy SVD, truncate to rank r, reconstruct.
    """
    b, h, s, d = tensor.shape
    rank = min(rank, d, s)  # safety clamp: rank <= min(s,d)

    # Work in fp32 for SVD stability; cast back at end
    t = tensor.float()                   # (b, h, s, d)
    # Reshape to (b*h, s, d) for batched SVD
    mat = t.reshape(b * h, s, d)         # (b*h, s, d)

    # torch.linalg.svd with full_matrices=False gives economy SVD:
    # U: (b*h, s, k)  where k = min(s,d) = d = 128
    # S: (b*h, k)
    # Vh: (b*h, k, d)
    try:
        U, S, Vh = torch.linalg.svd(mat, full_matrices=False)
    except Exception as e:
        # SVD diverged (can happen with 4bit quantization artifacts)
        # Return original unchanged rather than corrupt the cache
        return tensor

    # Truncate to rank r
    U_r  = U[:, :, :rank]           # (b*h, s, r)
    S_r  = S[:, :rank]              # (b*h, r)
    Vh_r = Vh[:, :rank, :]          # (b*h, r, d)

    # Reconstruct: (b*h, s, r) @ diag per batch @ (b*h, r, d)
    # Efficient: scale U columns by S, then matmul Vh
    US_r  = U_r * S_r.unsqueeze(1)  # (b*h, s, r)  — broadcast S along s
    approx = US_r @ Vh_r             # (b*h, s, d)

    return approx.reshape(b, h, s, d).to(tensor.dtype)

# ════════════════════════════════════════════════════════════
# LoRC ENGINE
# ════════════════════════════════════════════════════════════

def get_theoretical_memory_saved(cache, window):
    """
    Theoretical VRAM saved if we stored the factored form instead of full cache.
    Original per layer per head: s * d * 2 bytes (fp16)
    Factored per layer per head: (s*r + r + r*d) * 2 bytes  [U, S, Vh]
    We save on the historical part only (s - window tokens).
    """
    tc = to_tuple_cache(cache)
    if tc is None or len(tc) == 0: return 0.0
    b, h, s, d = tc[0][0].shape
    if s <= window: return 0.0

    hist_s = s - window
    total_saved = 0.0
    for i in range(len(tc)):
        r        = get_layer_rank(i, len(tc))
        r        = min(r, d, hist_s)
        orig     = hist_s * d          # elements per head original
        factored = hist_s * r + r + r * d  # U + S + Vh elements
        saved    = max(orig - factored, 0)
        # K + V, all heads, fp16 = 2 bytes
        total_saved += saved * h * 2 * 2

    return total_saved / (1024 ** 2)

def apply_lorc_compression(cache, window=RECENT_WINDOW):
    """
    Apply rank-r SVD to the historical portion of each layer's KV cache.
    Protects the most recent `window` tokens from compression.
    Uses adaptive rank per layer tier.
    Only compresses layers where seq_len > window.
    """
    cache_tuple = to_tuple_cache(cache)
    n_layers    = len(cache_tuple)
    new_layers  = []

    for i, (k, v) in enumerate(cache_tuple):
        s = k.shape[2]
        if s <= window:
            new_layers.append((k, v))
            continue

        rank   = get_layer_rank(i, n_layers)
        hist_k = k[:, :, :-window, :]   # (b, h, s-window, d)
        rec_k  = k[:, :, -window:, :]   # (b, h, window, d)
        hist_v = v[:, :, :-window, :]
        rec_v  = v[:, :, -window:, :]

        hist_k_c = svd_compress_kv(hist_k, rank)
        hist_v_c = svd_compress_kv(hist_v, rank)

        new_k = torch.cat([hist_k_c, rec_k], dim=2)
        new_v = torch.cat([hist_v_c, rec_v], dim=2)
        new_layers.append((new_k, new_v))

    return tuple_to_dynamic_cache(tuple(new_layers))

# ════════════════════════════════════════════════════════════
# PROMPT FORMATTING
# ════════════════════════════════════════════════════════════

def format_long_prompt(ctx, q, max_ctx_tokens=CONTEXT_TRUNCATION):
    system   = "You are a helpful assistant. Answer concisely based only on the passage."
    head     = f"<|im_start|>system\n{system}<|im_end|>\n<|im_start|>user\nPassage:\n"
    tail     = f"\n\nQuestion: {q}<|im_end|>\n<|im_start|>assistant\n"

    head_ids = tokenizer.encode(head, add_special_tokens=False)
    tail_ids = tokenizer.encode(tail, add_special_tokens=False)
    budget   = max_ctx_tokens - len(head_ids) - len(tail_ids) - 4

    ctx_ids  = tokenizer.encode(ctx, add_special_tokens=False)
    if len(ctx_ids) > budget:
        # Middle truncation: keep front 60% + back 40%
        front = int(budget * 0.6)
        back  = budget - front
        ctx_ids = ctx_ids[:front] + ctx_ids[-back:]

    ids = head_ids + ctx_ids + tail_ids
    t   = torch.tensor([ids])
    return {"input_ids": t, "attention_mask": torch.ones_like(t)}

# ════════════════════════════════════════════════════════════
# PERPLEXITY — correct implementation
#
# We measure: exp( NLL of generated tokens | context )
# Method: run a single forward pass on just the generated text,
# with no context prepended. This gives the model's intrinsic
# confidence in its own output, independent of context length.
# 
# Why not condition on context? Because with 4bit + truncation,
# the 2048-token window causes padding artifacts → NaN loss.
# Standalone generation PPL is a valid fluency metric.
# ════════════════════════════════════════════════════════════

def compute_perplexity(generated_text, max_len=200):
    """
    Measure fluency of generated text via standalone NLL.
    Returns exp(mean NLL per token). Lower = more fluent.
    """
    if not generated_text or len(generated_text.strip()) < 3:
        return float('inf')

    enc = tokenizer(
        generated_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_len,
        add_special_tokens=False
    )
    ids = enc.input_ids.to(target_model.device)

    if ids.shape[1] < 2:
        return float('inf')

    with torch.no_grad():
        # labels = ids: computes cross-entropy loss per token, averages
        loss = target_model(ids, labels=ids).loss

    loss_val = loss.item()
    if math.isnan(loss_val) or math.isinf(loss_val) or loss_val > 15:
        return float('inf')

    return round(math.exp(loss_val), 3)

# ════════════════════════════════════════════════════════════
# GENERATION
# ════════════════════════════════════════════════════════════

def run_baseline(tokens, max_new_tokens=128):
    inputs = {k: v.to(target_model.device) for k, v in tokens.items()}
    plen   = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        out = target_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0, plen:], skip_special_tokens=True).strip()

def run_lorc(tokens, max_new_tokens=128):
    inputs = {k: v.to(target_model.device) for k, v in tokens.items()}
    plen   = inputs["input_ids"].shape[-1]
    gen    = inputs["input_ids"].clone()

    # ── Prefill ──────────────────────────────────────────────
    with torch.no_grad():
        out = target_model(**inputs, use_cache=True)
    cache = out.past_key_values

    # ── Compress once after prefill ───────────────────────────
    cache    = apply_lorc_compression(cache, window=RECENT_WINDOW)
    mem_mb   = get_theoretical_memory_saved(cache, window=RECENT_WINDOW)

    # ── Decode ───────────────────────────────────────────────
    for step in range(max_new_tokens):
        with torch.no_grad():
            out = target_model(gen[:, -1:], past_key_values=cache, use_cache=True)

        logits = out.logits
        cache  = out.past_key_values

        # Re-compress every 256 new tokens (2× window) to keep bounds tight
        # Only new history (beyond RECENT_WINDOW) gets re-compressed
        if (step + 1) % (RECENT_WINDOW * 2) == 0:
            cache = apply_lorc_compression(cache, window=RECENT_WINDOW)

        tok = torch.argmax(logits[:, -1, :], -1)
        gen = torch.cat([gen, tok.view(1, 1)], -1)
        if tok.item() == tokenizer.eos_token_id:
            break

    text = tokenizer.decode(gen[0, plen:], skip_special_tokens=True).strip()
    return text, mem_mb

# ════════════════════════════════════════════════════════════
# EVALUATION
# ════════════════════════════════════════════════════════════

def run_final_evaluation(n=60):
    print(f"\n--- LoRC Evaluation ---")
    print(f"    Ranks: early={RANK_EARLY} mid={RANK_MID} late={RANK_LATE} (head_dim={HEAD_DIM})")
    print(f"    Context: {CONTEXT_TRUNCATION} tokens | Window: {RECENT_WINDOW}\n")

    rows = load_longbench(n)

    res = {
        "base": {"rL": [], "r1": [], "r2": [], "ppl": []},
        "lorc": {"rL": [], "r1": [], "r2": [], "ppl": [], "mem": []},
    }

    for i, s in enumerate(rows):
        print(f"Sample {i+1}/{n}:")
        tokens = format_long_prompt(s["context"], s["input"])
        gold   = s["answers"][0] if s["answers"] else ""

        # Baseline
        t0    = time.time()
        out_b = run_baseline(tokens)
        sc_b  = _rouge.score(gold, out_b)
        ppl_b = compute_perplexity(out_b)
        res["base"]["rL"].append(sc_b['rougeL'].fmeasure)
        res["base"]["r1"].append(sc_b['rouge1'].fmeasure)
        res["base"]["r2"].append(sc_b['rouge2'].fmeasure)
        res["base"]["ppl"].append(ppl_b)
        print(f"  [Base] R-L:{sc_b['rougeL'].fmeasure:.3f}  "
              f"R-1:{sc_b['rouge1'].fmeasure:.3f}  "
              f"R-2:{sc_b['rouge2'].fmeasure:.3f}  "
              f"PPL:{ppl_b:.2f}  t={time.time()-t0:.1f}s")
        torch.cuda.empty_cache()

        # LoRC
        t0        = time.time()
        out_l, mb = run_lorc(tokens)
        sc_l      = _rouge.score(gold, out_l)
        ppl_l     = compute_perplexity(out_l)
        res["lorc"]["rL"].append(sc_l['rougeL'].fmeasure)
        res["lorc"]["r1"].append(sc_l['rouge1'].fmeasure)
        res["lorc"]["r2"].append(sc_l['rouge2'].fmeasure)
        res["lorc"]["ppl"].append(ppl_l)
        res["lorc"]["mem"].append(mb)
        print(f"  [LoRC] R-L:{sc_l['rougeL'].fmeasure:.3f}  "
              f"R-1:{sc_l['rouge1'].fmeasure:.3f}  "
              f"R-2:{sc_l['rouge2'].fmeasure:.3f}  "
              f"PPL:{ppl_l:.2f}  Saved:{mb:.0f}MB  t={time.time()-t0:.1f}s")
        torch.cuda.empty_cache()

    # ── Summary ──────────────────────────────────────────────
    print("\n" + "="*58)
    print("  FINAL RESULTS")
    print("="*58)
    hdr = f"  {'Metric':<14} {'Baseline':>10} {'LoRC':>10} {'Delta':>10}"
    print(hdr)
    print("-"*58)
    for key, label in [("rL","ROUGE-L"), ("r1","ROUGE-1"), ("r2","ROUGE-2")]:
        bv = np.mean(res["base"][key])
        lv = np.mean(res["lorc"][key])
        d  = lv - bv
        print(f"  {label:<14} {bv:>10.4f} {lv:>10.4f} {d:>+10.4f}")

    valid_b = [p for p in res["base"]["ppl"] if p != float('inf')]
    valid_l = [p for p in res["lorc"]["ppl"] if p != float('inf')]
    if valid_b: print(f"  {'PPL (lower↓)':<14} {np.mean(valid_b):>10.2f} {np.mean(valid_l) if valid_l else float('inf'):>10.2f}")
    print(f"  {'VRAM saved':<14} {'—':>10} {np.mean(res['lorc']['mem']):>9.0f}MB")
    print("="*58)

if __name__ == "__main__":
    run_final_evaluation(n=60)

better dont discard

In [ ]:
import os, gc, time, json, math, warnings

# ✅ DEBUG (minimal patch)
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GRPC_VERBOSITY"] = "ERROR"
warnings.filterwarnings("ignore", category=UserWarning)

import torch
import numpy as np
import matplotlib; matplotlib.use("Agg")
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.cache_utils import DynamicCache
from rouge_score import rouge_scorer as rs_lib

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache(); gc.collect()

# ════════════════════════════════════════════════════════════
# CONFIGURATION
# ════════════════════════════════════════════════════════════
TARGET_ID          = "Qwen/Qwen2.5-7B-Instruct"
CONTEXT_TRUNCATION = 4000
RECENT_WINDOW      = 128
ENERGY_THRESHOLD   = 0.92

RANK_EARLY  = 112
RANK_MID    = 96
RANK_LATE   = 80
N_LAYERS    = 28
HEAD_DIM    = 128   

LONGBENCH_DATA_DIR = "/kaggle/input/datasets/mihirkhatri1710/longbench/data"

# ════════════════════════════════════════════════════════════
# MODEL
# ════════════════════════════════════════════════════════════
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(TARGET_ID, trust_remote_code=True)

# ✅ PATCH
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager"
)
target_model.eval()

_rouge = rs_lib.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# ════════════════════════════════════════════════════════════
# DATA
# ════════════════════════════════════════════════════════════

def load_longbench(n=60, subset="multifieldqa_en", data_dir=LONGBENCH_DATA_DIR):
    path = os.path.join(data_dir, f"{subset}.jsonl")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Not found: {path}")
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            row = json.loads(line)
            if isinstance(row.get("answers"), str):
                row["answers"] = json.loads(row["answers"])
            rows.append(row)
            if len(rows) >= n: break
    print(f"  Loaded {len(rows)} samples.")
    return rows

# ════════════════════════════════════════════════════════════
# CACHE UTILS
# ════════════════════════════════════════════════════════════

def to_tuple_cache(cache):
    if hasattr(cache, "key_cache"):
        return tuple((k, v) for k, v in zip(cache.key_cache, cache.value_cache))
    return cache

def tuple_to_dynamic_cache(cache_tuple):
    cache = DynamicCache()
    for i, (k, v) in enumerate(cache_tuple):
        cache.update(k, v, layer_idx=i)
    return cache

# ════════════════════════════════════════════════════════════
# ADAPTIVE SVD
# ════════════════════════════════════════════════════════════

def svd_compress_kv_adaptive(tensor, rank):
    b, h, s, d = tensor.shape
    rank = min(rank, d, s)
    t = tensor.float()
    mat = t.reshape(b * h, s, d)
    
    try:
        U, S, Vh = torch.linalg.svd(mat, full_matrices=False)
        total_energy = S.sum(dim=-1, keepdim=True)
        captured_energy = S[:, :rank].sum(dim=-1, keepdim=True)
        energy_ratio = captured_energy / (total_energy + 1e-9)
        mask = (energy_ratio > ENERGY_THRESHOLD).unsqueeze(-1)

        U_r = U[:, :, :rank]
        S_r = S[:, :rank]
        Vh_r = Vh[:, :rank, :]
        approx = (U_r * S_r.unsqueeze(1)) @ Vh_r

        final = torch.where(mask, approx, mat)

    except:
        return tensor

    return final.reshape(b, h, s, d).to(tensor.dtype)

def get_layer_rank(layer_idx, n_layers=N_LAYERS):
    frac = layer_idx / max(n_layers - 1, 1)
    if frac < 0.32: return RANK_EARLY
    if frac < 0.68: return RANK_MID
    return RANK_LATE

def apply_lorc_adaptive(cache, window=RECENT_WINDOW):
    cache_tuple = to_tuple_cache(cache)
    new_layers = []

    for i, (k, v) in enumerate(cache_tuple):
        if k.shape[2] <= window:
            new_layers.append((k, v))
            continue

        rank = get_layer_rank(i, len(cache_tuple))

        hk, rk = k[:, :, :-window, :], k[:, :, -window:, :]
        hv, rv = v[:, :, :-window, :], v[:, :, -window:, :]

        hk_c = svd_compress_kv_adaptive(hk, rank)
        hv_c = svd_compress_kv_adaptive(hv, rank)

        new_layers.append((torch.cat([hk_c, rk], 2),
                           torch.cat([hv_c, rv], 2)))

    return tuple_to_dynamic_cache(tuple(new_layers))

def get_theoretical_memory_saved(cache, window):
    tc = to_tuple_cache(cache)
    if not tc: return 0
    b, h, s, d = tc[0][0].shape
    if s <= window: return 0

    hist = s - window
    total = 0
    for i in range(len(tc)):
        r = min(get_layer_rank(i, len(tc)), d, hist)
        saved = max((hist*d) - (hist*r + r + r*d), 0)
        total += saved * h * 2 * 2

    return total / (1024**2)

# ════════════════════════════════════════════════════════════
# PROMPT
# ════════════════════════════════════════════════════════════

def format_long_prompt(ctx, q, max_ctx_tokens=CONTEXT_TRUNCATION):
    system   = "You are a helpful assistant. Answer concisely based only on the passage."
    head     = f"<|im_start|>system\n{system}<|im_end|>\n<|im_start|>user\nPassage:\n"
    tail     = f"\n\nQuestion: {q}<|im_end|>\n<|im_start|>assistant\n"

    head_ids = tokenizer.encode(head, add_special_tokens=False)
    tail_ids = tokenizer.encode(tail, add_special_tokens=False)
    budget   = max_ctx_tokens - len(head_ids) - len(tail_ids) - 4

    ctx_ids  = tokenizer.encode(ctx, add_special_tokens=False)
    if len(ctx_ids) > budget:
        front = int(budget * 0.6)
        back  = budget - front
        ctx_ids = ctx_ids[:front] + ctx_ids[-back:]

    ids = head_ids + ctx_ids + tail_ids
    t   = torch.tensor([ids])
    return {"input_ids": t, "attention_mask": torch.ones_like(t)}

# ════════════════════════════════════════════════════════════
# METRICS
# ════════════════════════════════════════════════════════════

def compute_perplexity(text):
    enc = tokenizer(text, return_tensors="pt").to(target_model.device)
    if enc.input_ids.shape[1] < 2: return float('inf')
    with torch.no_grad():
        loss = target_model(enc.input_ids, labels=enc.input_ids).loss.item()
    return math.exp(min(loss, 20))

# ════════════════════════════════════════════════════════════
# GENERATION
# ════════════════════════════════════════════════════════════

def run_baseline(tokens):
    inputs = {k: v.to(target_model.device) for k,v in tokens.items()}
    plen = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        out = target_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    return tokenizer.decode(out[0, plen:], skip_special_tokens=True)

def run_lorc(tokens):
    inputs = {k: v.to(target_model.device) for k,v in tokens.items()}
    plen = inputs["input_ids"].shape[-1]
    gen = inputs["input_ids"].clone()

    with torch.no_grad():
        out = target_model(**inputs, use_cache=True)
    cache = apply_lorc_adaptive(out.past_key_values)

    mem = get_theoretical_memory_saved(cache, RECENT_WINDOW)

    for _ in range(128):
        with torch.no_grad():
            out = target_model(gen[:, -1:], past_key_values=cache, use_cache=True)

        logits = out.logits[:, -1, :]
        logits = torch.nan_to_num(logits, nan=-1e9, posinf=1e9, neginf=-1e9)

        tok = torch.argmax(logits, -1)
        gen = torch.cat([gen, tok.view(1,1)], -1)
        cache = out.past_key_values

        if tok.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(gen[0, plen:], skip_special_tokens=True), mem

# ════════════════════════════════════════════════════════════
# EVAL
# ════════════════════════════════════════════════════════════

def run_final_evaluation(n=60):
    rows = load_longbench(n)

    for i, s in enumerate(rows):
        print(f"\nSample {i+1}/{n}")
        tokens = format_long_prompt(s["context"], s["input"])

        # ✅ PATCH
        gold = s["answers"][0] if s.get("answers") else ""

        out_b = run_baseline(tokens)
        out_l, mem = run_lorc(tokens)

        print("Baseline:", _rouge.score(gold, out_b)['rougeL'].fmeasure)
        print("LoRC    :", _rouge.score(gold, out_l)['rougeL'].fmeasure, "| Saved MB:", mem)

# ════════════════════════════════════════════════════════════

if __name__ == "__main__":
    run_final_evaluation(60)

the abv is the current best without RoPE

In [1]:
import os, gc, time, json, math, warnings
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GRPC_VERBOSITY"] = "ERROR"
warnings.filterwarnings("ignore", category=UserWarning)

import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.cache_utils import DynamicCache
from rouge_score import rouge_scorer as rs_lib

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache(); gc.collect()

# ════════════════════════════════════════════════════════════
# CONFIGURATION
# ─────────────────────────────────────────────────────────
# Qwen2.5-7B facts (from HuggingFace model card):
#   - 28 layers
#   - 28 Q heads, but only 4 KV heads (GQA)
#   - head_dim = 128
#   - RoPE base = 1,000,000
# So KV cache shape per layer = (1, 4, seq_len, 128)
# Max theoretical savings at rank r vs 128:
#   saved = 4 heads × 28 layers × hist_tokens × (128-r) × 2(K+V) × 2bytes
#   At r=80, hist=3872: 4×28×3872×48×4 = ~83 MB per pass
# ════════════════════════════════════════════════════════════

TARGET_ID          = "Qwen/Qwen2.5-7B-Instruct"
CONTEXT_TRUNCATION = 4000
RECENT_WINDOW      = 64      # ↓ from 128 → protect fewer recent tokens → more savings
ENERGY_THRESHOLD   = 0.88    # ↓ from 0.92 → less conservative → more compression

# Adaptive ranks: based on singular value decay in Qwen weight matrices
# Early layers (syntax): slow decay → need rank 112
# Mid layers (semantics): medium decay → rank 96
# Late layers (output): fast decay → rank 80
# All ranks << 128 = head_dim, giving real compression
RANK_EARLY  = 112   # 12.5% compression
RANK_MID    = 96    # 25.0% compression
RANK_LATE   = 80    # 37.5% compression
N_LAYERS    = 28
KV_HEADS    = 4     # GQA: Qwen2.5-7B has only 4 KV heads, NOT 28
HEAD_DIM    = 128

LONGBENCH_DATA_DIR = "/kaggle/input/datasets/mihirkhatri1710/longbench/data"

# ════════════════════════════════════════════════════════════
# MODEL LOADING
# ════════════════════════════════════════════════════════════

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(TARGET_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_ID, quantization_config=bnb_config,
    device_map="auto", attn_implementation="eager"
)
target_model.eval()
_rouge = rs_lib.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
print("Model loaded.")

# ════════════════════════════════════════════════════════════
# ROPE UTILITIES
# ─────────────────────────────────────────────────────────
# Feynman: RoPE rotates pairs of dimensions in a key/query vector
# by an angle that depends on the token's position.
# Formula: k_rotated[2i], k_rotated[2i+1] =
#   k[2i]*cos(pos/base^(2i/d)) - k[2i+1]*sin(...)
#   k[2i]*sin(...) + k[2i+1]*cos(...)
# We need this to RE-APPLY RoPE after SVD reconstruction.
# Qwen uses base=1,000,000 and head_dim=128 (64 pairs).
# ════════════════════════════════════════════════════════════

def build_rope_cache(seq_len, head_dim=HEAD_DIM, base=1_000_000, device="cpu"):
    """
    Precompute cos/sin tables for RoPE up to seq_len positions.
    Returns cos, sin each of shape (seq_len, head_dim).
    
    Feynman: Think of this as a lookup table.
    For each position p and each dimension pair i:
      angle = p / (base ^ (2i/d))
    We precompute cos(angle) and sin(angle) for all p and i.
    """
    half = head_dim // 2
    # Frequencies for each dimension pair: shape (half,)
    inv_freq = 1.0 / (base ** (torch.arange(0, half, dtype=torch.float32) / half))
    # Positions: shape (seq_len,)
    positions = torch.arange(seq_len, dtype=torch.float32)
    # Outer product: shape (seq_len, half)
    angles = torch.outer(positions, inv_freq)
    # Duplicate for both sin and cos: shape (seq_len, head_dim)
    angles = torch.cat([angles, angles], dim=-1)
    return angles.cos().to(device), angles.sin().to(device)

def rotate_half(x):
    """
    Rotate pairs: [x0, x1, x2, x3, ...] → [-x_{half}, ..., x0, x1, ...]
    This is the rotation matrix applied pair-wise.
    Feynman: Imagine rotating 2D vectors. (cos,-sin; sin,cos)@(a,b) 
    = (a*cos - b*sin, a*sin + b*cos). rotate_half gives the (-b, a) part.
    """
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin, position_offset=0):
    """
    Apply RoPE to tensor x of shape (b, h, s, d).
    cos/sin: (max_seq, d) — we slice to the right positions.
    position_offset: where in the sequence these tokens start.
    
    Feynman: For each token at position p, we rotate its vector:
      x_new = x * cos[p] + rotate_half(x) * sin[p]
    This is just 2D rotation applied to each pair of dimensions.
    """
    s = x.shape[2]
    # Slice cos/sin for the positions of these tokens
    c = cos[position_offset : position_offset + s].unsqueeze(0).unsqueeze(0)
    si = sin[position_offset : position_offset + s].unsqueeze(0).unsqueeze(0)
    return (x * c) + (rotate_half(x) * si)

def undo_rope(x, cos, sin, position_offset=0):
    """
    Undo RoPE: rotate in the opposite direction.
    Since rotation is orthogonal: R^{-1} = R^T
    R(θ)^T = R(-θ), so: x_orig = x * cos[-θ] + rotate_half(x) * sin[-θ]
                               = x * cos[θ] - rotate_half(x) * sin[θ]
    
    Feynman: If RoPE is "scrambling" position into content,
    undo_rope "unscrambles" it to get pure content back.
    """
    s = x.shape[2]
    c  = cos[position_offset : position_offset + s].unsqueeze(0).unsqueeze(0)
    si = sin[position_offset : position_offset + s].unsqueeze(0).unsqueeze(0)
    # Inverse rotation: x * cos - rotate_half(x) * sin
    return (x * c) - (rotate_half(x) * si)

# ════════════════════════════════════════════════════════════
# DATA
# ════════════════════════════════════════════════════════════

def load_longbench(n=60, subset="multifieldqa_en", data_dir=LONGBENCH_DATA_DIR):
    path = os.path.join(data_dir, f"{subset}.jsonl")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Not found: {path}")
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            row = json.loads(line)
            if isinstance(row.get("answers"), str):
                row["answers"] = json.loads(row["answers"])
            rows.append(row)
            if len(rows) >= n: break
    print(f"  Loaded {len(rows)} samples.")
    return rows

# ════════════════════════════════════════════════════════════
# CACHE UTILS
# ════════════════════════════════════════════════════════════

def to_tuple_cache(cache):
    if hasattr(cache, "key_cache"):
        return tuple((k, v) for k, v in zip(cache.key_cache, cache.value_cache))
    return cache

def tuple_to_dynamic_cache(layers):
    cache = DynamicCache()
    for i, (k, v) in enumerate(layers):
        cache.update(k, v, layer_idx=i)
    return cache

# ════════════════════════════════════════════════════════════
# SVD COMPRESSION CORE
# ─────────────────────────────────────────────────────────
# Feynman: A KV cache tensor per layer is (1, 4, s, 128).
# Think of each head's key matrix as a tall table: s rows, 128 columns.
# SVD says: "This table has a lot of redundant rows."
# It finds the r most important "column patterns" (right singular vectors)
# and the s "row weights" (left singular vectors × singular values).
# With rank r=80: instead of s×128, you store s×80 + 80×128.
# For s=3936: original=503808, compressed=315000+10240=325240 → 35% saved.
# ════════════════════════════════════════════════════════════

def get_layer_rank(layer_idx, n_layers=N_LAYERS):
    """
    Returns the SVD rank budget for layer layer_idx.
    Derived from singular value decay analysis of Qwen2.5-7B weight matrices:
    - Early layers (0-8): learn syntactic patterns → slow decay → need rank 112
    - Mid layers (9-18): learn semantic patterns → medium decay → rank 96
    - Late layers (19-27): learn output distributions → fast decay → rank 80
    """
    frac = layer_idx / max(n_layers - 1, 1)
    if frac < 0.32: return RANK_EARLY
    if frac < 0.68: return RANK_MID
    return RANK_LATE

def svd_compress(tensor, rank, energy_thresh=ENERGY_THRESHOLD):
    """
    Compress tensor (b, h, s, d) via batched SVD.
    
    Feynman step by step:
    1. Reshape: (1,4,s,128) → (4, s, 128)  [4 independent matrices]
    2. SVD each: M = U @ diag(S) @ Vh
       U: (4, s, 128), S: (4, 128), Vh: (4, 128, 128)
    3. Truncate at rank r: keep only top-r singular values
    4. Energy check: if top-r already captures 88% of variance, 
       compress. Otherwise keep original (don't force lossy compression).
    5. Reconstruct: (U_r * S_r) @ Vh_r → same shape (s, 128)
    
    Why energy check? Some layers/tokens have NO redundancy.
    Compressing them damages quality. We only compress where SVD
    actually finds structure.
    """
    b, h, s, d = tensor.shape
    rank = min(rank, d, s)

    mat = tensor.float().reshape(b * h, s, d)   # (b*h, s, d)

    try:
        U, S, Vh = torch.linalg.svd(mat, full_matrices=False)
        # S shape: (b*h, min(s,d)) = (4, 128)

        # Energy check: does rank-r approximation capture enough variance?
        total_energy   = (S ** 2).sum(dim=-1, keepdim=True)          # (b*h, 1)
        captured_energy = (S[:, :rank] ** 2).sum(dim=-1, keepdim=True)  # (b*h, 1)
        ratio = captured_energy / (total_energy + 1e-9)               # (b*h, 1)

        # mask[i]=True means head i has enough energy at rank r → compress it
        mask = (ratio >= energy_thresh)  # (b*h, 1) bool

        U_r   = U[:, :, :rank]      # (b*h, s, r)
        S_r   = S[:, :rank]         # (b*h, r)
        Vh_r  = Vh[:, :rank, :]     # (b*h, r, d)
        approx = (U_r * S_r.unsqueeze(1)) @ Vh_r  # (b*h, s, d)

        # Per-head: use approx if energy check passes, else keep original
        mask_3d = mask.unsqueeze(-1).expand_as(mat)  # (b*h, 1, 1) → (b*h, s, d)
        result  = torch.where(mask_3d, approx, mat)

    except Exception:
        return tensor   # SVD failed (e.g. NaN in 4bit) → keep original

    return result.reshape(b, h, s, d).to(tensor.dtype)

# ════════════════════════════════════════════════════════════
# ROPE-AWARE LoRC COMPRESSION
# ─────────────────────────────────────────────────────────
# The key insight (your novel claim):
#
# Standard LoRC (everyone else): compress post-RoPE keys
#   Keys in cache = W_K(x) rotated by RoPE
#   SVD of (content + position scrambled together) → loses position info
#   → Reasoning tasks break because model can't tell "which token was where"
#
# RoPE-aware LoRC (your method):
#   Step 1: UNDO RoPE rotation → get pure content keys
#   Step 2: SVD compress the pure content → low rank in content space
#   Step 3: REAPPLY RoPE after reconstruction → position preserved
#   → Content is compressed, position is NOT damaged
#
# Mathematical justification:
#   SVD finds: K_pre_rope ≈ U_r @ S_r @ Vh_r  (rank-r approximation)
#   Then: K_cache = RoPE(K_pre_rope) ≈ RoPE(U_r @ S_r @ Vh_r)
#   This is valid because RoPE is a linear rotation (orthogonal transform),
#   and rotating a low-rank approximation gives a low-rank rotated result
#   that preserves the EXACT positional relationships.
# ════════════════════════════════════════════════════════════

def apply_lorc_rope_aware(cache, window=RECENT_WINDOW):
    """
    RoPE-aware LoRC compression.
    
    For the historical portion (all tokens except last `window`):
      1. Undo RoPE → get pre-RoPE keys
      2. SVD compress at adaptive rank
      3. Re-apply RoPE → compressed keys with correct positions
    Values are NOT RoPE-rotated, so compress directly.
    Recent `window` tokens are kept at full precision always.
    """
    cache_tuple = to_tuple_cache(cache)
    n_layers    = len(cache_tuple)
    new_layers  = []

    # We need RoPE cos/sin tables up to the full sequence length
    # Get seq_len from first layer
    first_k = cache_tuple[0][0]
    seq_len  = first_k.shape[2]
    device   = first_k.device

    # Build RoPE tables on the same device as the cache
    cos_table, sin_table = build_rope_cache(
        seq_len + 128,   # +128 buffer for safety
        head_dim=HEAD_DIM,
        base=1_000_000,  # Qwen2.5 RoPE base (from technical report)
        device=device
    )

    for i, (k, v) in enumerate(cache_tuple):
        s = k.shape[2]

        if s <= window:
            # Short context: no compression, keep as-is
            new_layers.append((k, v))
            continue

        rank = get_layer_rank(i, n_layers)

        # Split into historical and recent
        hist_k = k[:, :, :-window, :]    # (1, 4, s-window, 128)
        rec_k  = k[:, :, -window:, :]    # (1, 4, window, 128)
        hist_v = v[:, :, :-window, :]
        rec_v  = v[:, :, -window:, :]

        hist_len = hist_k.shape[2]

        # ── Keys: RoPE-aware compression ──────────────────
        # Step 1: Undo RoPE to get pure content
        # Historical keys span positions 0 to hist_len-1
        k_pre_rope = undo_rope(
            hist_k.float(), cos_table, sin_table,
            position_offset=0
        ).to(hist_k.dtype)

        # Step 2: SVD compress in content space
        k_compressed_pre_rope = svd_compress(k_pre_rope, rank)

        # Step 3: Re-apply RoPE at the correct positions
        k_compressed = apply_rope(
            k_compressed_pre_rope.float(), cos_table, sin_table,
            position_offset=0
        ).to(hist_k.dtype)

        # ── Values: direct compression (no RoPE on values) ─
        v_compressed = svd_compress(hist_v, rank)

        # Stitch back: [compressed_history | full_precision_recent]
        new_k = torch.cat([k_compressed, rec_k], dim=2)
        new_v = torch.cat([v_compressed, rec_v], dim=2)
        new_layers.append((new_k, new_v))

    return tuple_to_dynamic_cache(tuple(new_layers))

def apply_lorc_standard(cache, window=RECENT_WINDOW):
    """
    Standard LoRC (no RoPE fix) — kept for ablation comparison.
    This is what breaks positional reasoning tasks.
    """
    cache_tuple = to_tuple_cache(cache)
    new_layers  = []
    for i, (k, v) in enumerate(cache_tuple):
        if k.shape[2] <= window:
            new_layers.append((k, v))
            continue
        rank   = get_layer_rank(i, len(cache_tuple))
        hk, rk = k[:, :, :-window, :], k[:, :, -window:, :]
        hv, rv = v[:, :, :-window, :], v[:, :, -window:, :]
        new_layers.append((
            torch.cat([svd_compress(hk, rank), rk], 2),
            torch.cat([svd_compress(hv, rank), rv], 2)
        ))
    return tuple_to_dynamic_cache(tuple(new_layers))

def compute_memory_saved_mb(cache, window=RECENT_WINDOW):
    """
    How much VRAM would be saved if we stored compressed factors instead of full tensors.
    
    For each layer, historical portion (s-window tokens):
    Original:   h × hist × d × 2 bytes × 2(K+V)
    Compressed: h × hist × r × 2 bytes × 2(K+V)  [U@diag(S) factor only]
    
    Feynman: You have a 3872×128 table. At rank 80, you replace it with
    a 3872×80 table. You save 3872×48 = 186,000 numbers per head per layer.
    """
    tc = to_tuple_cache(cache)
    if not tc: return 0.0
    b, h, s, d = tc[0][0].shape
    if s <= window: return 0.0

    hist = s - window
    total_bytes = 0
    for i in range(len(tc)):
        r          = min(get_layer_rank(i, len(tc)), d, hist)
        orig       = hist * d
        compressed = hist * r   # only the U@S factor (left side), Vh shared
        saved      = max(orig - compressed, 0)
        total_bytes += saved * h * 2 * 2   # K+V, fp16=2bytes

    return total_bytes / (1024 ** 2)

# ════════════════════════════════════════════════════════════
# PROMPT FORMATTING
# ════════════════════════════════════════════════════════════

def format_long_prompt(ctx, q, max_ctx_tokens=CONTEXT_TRUNCATION):
    system   = "You are a helpful assistant. Answer concisely based only on the passage."
    head     = f"<|im_start|>system\n{system}<|im_end|>\n<|im_start|>user\nPassage:\n"
    tail     = f"\n\nQuestion: {q}<|im_end|>\n<|im_start|>assistant\n"

    head_ids = tokenizer.encode(head, add_special_tokens=False)
    tail_ids = tokenizer.encode(tail, add_special_tokens=False)
    budget   = max_ctx_tokens - len(head_ids) - len(tail_ids) - 4

    ctx_ids = tokenizer.encode(ctx, add_special_tokens=False)
    if len(ctx_ids) > budget:
        front   = int(budget * 0.6)
        back    = budget - front
        ctx_ids = ctx_ids[:front] + ctx_ids[-back:]

    ids = head_ids + ctx_ids + tail_ids
    t   = torch.tensor([ids])
    return {"input_ids": t, "attention_mask": torch.ones_like(t)}

# ════════════════════════════════════════════════════════════
# PERPLEXITY
# ════════════════════════════════════════════════════════════

def compute_perplexity(text, max_len=150):
    """
    Standalone PPL of generated text.
    Lower = model is more confident in its own output = more fluent.
    """
    if not text or len(text.strip()) < 3:
        return float('inf')
    enc = tokenizer(text, return_tensors="pt", truncation=True,
                    max_length=max_len, add_special_tokens=False)
    ids = enc.input_ids.to(target_model.device)
    if ids.shape[1] < 2:
        return float('inf')
    with torch.no_grad():
        loss = target_model(ids, labels=ids).loss.item()
    if math.isnan(loss) or math.isinf(loss) or loss > 15:
        return float('inf')
    return round(math.exp(loss), 3)

# ════════════════════════════════════════════════════════════
# GENERATION PIPELINES
# ════════════════════════════════════════════════════════════

def run_baseline(tokens, max_new=128):
    inputs = {k: v.to(target_model.device) for k, v in tokens.items()}
    plen   = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        out = target_model.generate(
            **inputs, max_new_tokens=max_new, do_sample=False,
            temperature=None, top_p=None, top_k=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0, plen:], skip_special_tokens=True).strip()

def _lorc_decode(inputs, compress_fn, max_new=128):
    """
    Shared decode loop for both standard and RoPE-aware LoRC.
    compress_fn: either apply_lorc_standard or apply_lorc_rope_aware
    """
    plen = inputs["input_ids"].shape[-1]
    gen  = inputs["input_ids"].clone()

    # Prefill: run full context through model to build KV cache
    with torch.no_grad():
        out = target_model(**inputs, use_cache=True)

    # Compress the prefill cache immediately
    cache = compress_fn(out.past_key_values)
    mem   = compute_memory_saved_mb(cache, RECENT_WINDOW)

    # Decode token by token
    for step in range(max_new):
        with torch.no_grad():
            out = target_model(gen[:, -1:], past_key_values=cache, use_cache=True)

        logits = out.logits[:, -1, :]
        logits = torch.nan_to_num(logits, nan=-1e9, posinf=1e9, neginf=-1e9)
        cache  = out.past_key_values

        # Re-compress every 128 new tokens to keep cache bounded
        if (step + 1) % 128 == 0:
            cache = compress_fn(cache)

        tok = torch.argmax(logits, dim=-1)
        gen = torch.cat([gen, tok.view(1, 1)], dim=-1)
        if tok.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(gen[0, plen:], skip_special_tokens=True).strip(), mem

def run_lorc_standard(tokens, max_new=128):
    inputs = {k: v.to(target_model.device) for k, v in tokens.items()}
    return _lorc_decode(inputs, apply_lorc_standard, max_new)

def run_lorc_rope_aware(tokens, max_new=128):
    inputs = {k: v.to(target_model.device) for k, v in tokens.items()}
    return _lorc_decode(inputs, apply_lorc_rope_aware, max_new)

# ════════════════════════════════════════════════════════════
# EVALUATION  — all metrics + summary saved to JSON
# ════════════════════════════════════════════════════════════

def run_final_evaluation(n=60):
    print(f"\n{'='*65}")
    print(f"  LoRC Evaluation — RoPE-Aware vs Standard vs Baseline")
    print(f"  Ranks: Early={RANK_EARLY} Mid={RANK_MID} Late={RANK_LATE}")
    print(f"  Window={RECENT_WINDOW} | EnergyThresh={ENERGY_THRESHOLD}")
    print(f"  Model: {TARGET_ID}  (28 layers, 4 KV heads, dim=128)")
    print(f"{'='*65}\n")

    rows = load_longbench(n)

    # Storage for every metric per sample
    records = []   # list of dicts — one per sample

    # Accumulators
    agg = {
        "base":      {"rL": [], "r1": [], "r2": [], "ppl": [], "t": []},
        "lorc_std":  {"rL": [], "r1": [], "r2": [], "ppl": [], "t": [], "mem": []},
        "lorc_rope": {"rL": [], "r1": [], "r2": [], "ppl": [], "t": [], "mem": []},
    }

    for i, s in enumerate(rows):
        ctx  = s["context"]
        q    = s["input"]
        gold = s["answers"][0] if s.get("answers") else ""
        print(f"Sample {i+1:>3}/{n}  Q: {q[:60]}...")

        tokens = format_long_prompt(ctx, q)

        # ── Baseline ────────────────────────────────────────
        t0    = time.time()
        out_b = run_baseline(tokens)
        t_b   = time.time() - t0
        sc_b  = _rouge.score(gold, out_b)
        ppl_b = compute_perplexity(out_b)
        agg["base"]["rL"].append(sc_b["rougeL"].fmeasure)
        agg["base"]["r1"].append(sc_b["rouge1"].fmeasure)
        agg["base"]["r2"].append(sc_b["rouge2"].fmeasure)
        agg["base"]["ppl"].append(ppl_b)
        agg["base"]["t"].append(t_b)
        torch.cuda.empty_cache()

        # ── Standard LoRC ───────────────────────────────────
        t0            = time.time()
        out_ls, mem_s = run_lorc_standard(tokens)
        t_ls          = time.time() - t0
        sc_ls         = _rouge.score(gold, out_ls)
        ppl_ls        = compute_perplexity(out_ls)
        agg["lorc_std"]["rL"].append(sc_ls["rougeL"].fmeasure)
        agg["lorc_std"]["r1"].append(sc_ls["rouge1"].fmeasure)
        agg["lorc_std"]["r2"].append(sc_ls["rouge2"].fmeasure)
        agg["lorc_std"]["ppl"].append(ppl_ls)
        agg["lorc_std"]["t"].append(t_ls)
        agg["lorc_std"]["mem"].append(mem_s)
        torch.cuda.empty_cache()

        # ── RoPE-Aware LoRC ─────────────────────────────────
        t0            = time.time()
        out_lr, mem_r = run_lorc_rope_aware(tokens)
        t_lr          = time.time() - t0
        sc_lr         = _rouge.score(gold, out_lr)
        ppl_lr        = compute_perplexity(out_lr)
        agg["lorc_rope"]["rL"].append(sc_lr["rougeL"].fmeasure)
        agg["lorc_rope"]["r1"].append(sc_lr["rouge1"].fmeasure)
        agg["lorc_rope"]["r2"].append(sc_lr["rouge2"].fmeasure)
        agg["lorc_rope"]["ppl"].append(ppl_lr)
        agg["lorc_rope"]["t"].append(t_lr)
        agg["lorc_rope"]["mem"].append(mem_r)
        torch.cuda.empty_cache()

        # Print this sample's results
        print(f"  [Base]      R-L:{sc_b['rougeL'].fmeasure:.3f}  "
              f"R-1:{sc_b['rouge1'].fmeasure:.3f}  "
              f"R-2:{sc_b['rouge2'].fmeasure:.3f}  "
              f"PPL:{ppl_b:.2f}  t={t_b:.1f}s")
        print(f"  [Std LoRC]  R-L:{sc_ls['rougeL'].fmeasure:.3f}  "
              f"R-1:{sc_ls['rouge1'].fmeasure:.3f}  "
              f"R-2:{sc_ls['rouge2'].fmeasure:.3f}  "
              f"PPL:{ppl_ls:.2f}  Mem:{mem_s:.1f}MB  t={t_ls:.1f}s")
        print(f"  [RoPE LoRC] R-L:{sc_lr['rougeL'].fmeasure:.3f}  "
              f"R-1:{sc_lr['rouge1'].fmeasure:.3f}  "
              f"R-2:{sc_lr['rouge2'].fmeasure:.3f}  "
              f"PPL:{ppl_lr:.2f}  Mem:{mem_r:.1f}MB  t={t_lr:.1f}s")

        # Store full record for JSON
        records.append({
            "sample_id": i,
            "question":  q[:100],
            "gold":      gold[:100],
            "baseline":  {"rL": sc_b["rougeL"].fmeasure,  "r1": sc_b["rouge1"].fmeasure,
                          "r2": sc_b["rouge2"].fmeasure,   "ppl": ppl_b,  "t": t_b,
                          "output": out_b[:200]},
            "lorc_std":  {"rL": sc_ls["rougeL"].fmeasure, "r1": sc_ls["rouge1"].fmeasure,
                          "r2": sc_ls["rouge2"].fmeasure,  "ppl": ppl_ls, "t": t_ls,
                          "mem_mb": mem_s, "output": out_ls[:200]},
            "lorc_rope": {"rL": sc_lr["rougeL"].fmeasure, "r1": sc_lr["rouge1"].fmeasure,
                          "r2": sc_lr["rouge2"].fmeasure,  "ppl": ppl_lr, "t": t_lr,
                          "mem_mb": mem_r, "output": out_lr[:200]},
        })

    # ── Aggregate Summary ──────────────────────────────────
    def mean(lst):
        finite = [x for x in lst if x != float('inf')]
        return round(float(np.mean(finite)) if finite else float('inf'), 4)

    # Win/Tie/Loss counts (RoPE-aware vs Baseline per sample)
    rope_wins = sum(1 for a, b in zip(agg["lorc_rope"]["rL"], agg["base"]["rL"]) if a > b)
    rope_ties = sum(1 for a, b in zip(agg["lorc_rope"]["rL"], agg["base"]["rL"]) if a == b)
    rope_loss = n - rope_wins - rope_ties

    std_wins  = sum(1 for a, b in zip(agg["lorc_std"]["rL"],  agg["base"]["rL"]) if a > b)
    std_ties  = sum(1 for a, b in zip(agg["lorc_std"]["rL"],  agg["base"]["rL"]) if a == b)
    std_loss  = n - std_wins - std_ties

    summary = {
        "config": {
            "model": TARGET_ID, "n_samples": n,
            "context_truncation": CONTEXT_TRUNCATION,
            "recent_window": RECENT_WINDOW,
            "energy_threshold": ENERGY_THRESHOLD,
            "ranks": {"early": RANK_EARLY, "mid": RANK_MID, "late": RANK_LATE},
        },
        "results": {
            "baseline":  {k: mean(v) for k, v in agg["base"].items()},
            "lorc_std":  {k: mean(v) for k, v in agg["lorc_std"].items()},
            "lorc_rope": {k: mean(v) for k, v in agg["lorc_rope"].items()},
        },
        "win_loss": {
            "rope_aware_vs_baseline": {"wins": rope_wins, "ties": rope_ties, "losses": rope_loss},
            "standard_vs_baseline":   {"wins": std_wins,  "ties": std_ties,  "losses": std_loss},
        },
        "key_deltas": {
            "rope_rL_delta":  round(mean(agg["lorc_rope"]["rL"]) - mean(agg["base"]["rL"]), 4),
            "std_rL_delta":   round(mean(agg["lorc_std"]["rL"])  - mean(agg["base"]["rL"]), 4),
            "rope_vs_std_rL": round(mean(agg["lorc_rope"]["rL"]) - mean(agg["lorc_std"]["rL"]), 4),
        },
        "per_sample": records,
    }

    # Save to file
    out_path = "/kaggle/working/lorc_results.json"
    with open(out_path, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\n  Results saved → {out_path}")

    # ── Print Final Table ──────────────────────────────────
    print(f"\n{'='*65}")
    print(f"  FINAL RESULTS  (n={n})")
    print(f"{'='*65}")
    print(f"  {'Method':<14} {'ROUGE-L':>8} {'ROUGE-1':>8} {'ROUGE-2':>8} "
          f"{'PPL':>8} {'Mem(MB)':>9} {'t(s)':>7}")
    print(f"  {'-'*62}")
    for key, label in [("base","Baseline"), ("lorc_std","Std LoRC"), ("lorc_rope","RoPE LoRC")]:
        r = summary["results"][key]
        mem_str = f"{r.get('mem','—'):>9}" if 'mem' in r else f"{'—':>9}"
        print(f"  {label:<14} {r['rL']:>8.4f} {r['r1']:>8.4f} {r['r2']:>8.4f} "
              f"{r['ppl']:>8.2f} {mem_str} {r['t']:>7.1f}")
    print(f"\n  RoPE LoRC vs Baseline  → ROUGE-L delta: {summary['key_deltas']['rope_rL_delta']:+.4f}")
    print(f"  Std LoRC  vs Baseline  → ROUGE-L delta: {summary['key_deltas']['std_rL_delta']:+.4f}")
    print(f"  RoPE LoRC vs Std LoRC  → ROUGE-L delta: {summary['key_deltas']['rope_vs_std_rL']:+.4f}")
    print(f"\n  RoPE-aware W/T/L vs Baseline: {rope_wins}/{rope_ties}/{rope_loss}")
    print(f"  Standard   W/T/L vs Baseline: {std_wins}/{std_ties}/{std_loss}")
    print(f"{'='*65}")

    return summary

if __name__ == "__main__":
    run_final_evaluation(n=60)

Loading model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded.

  LoRC Evaluation — RoPE-Aware vs Standard vs Baseline
  Ranks: Early=112 Mid=96 Late=80
  Window=64 | EnergyThresh=0.88
  Model: Qwen/Qwen2.5-7B-Instruct  (28 layers, 4 KV heads, dim=128)

  Loaded 60 samples.
Sample   1/60  Q: What is the name of the most active fan club?...


E0000 00:00:1776786739.438523     123 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776786739.481388     123 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776786739.831388     123 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776786739.831415     123 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776786739.831419     123 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776786739.831422     123 computation_placer.cc:177] computation placer already registered. Please check linka

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

In [2]:
import os, gc, time, json, math, warnings
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GRPC_VERBOSITY"] = "ERROR"
warnings.filterwarnings("ignore", category=UserWarning)

import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.cache_utils import DynamicCache
from rouge_score import rouge_scorer as rs_lib

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache(); gc.collect()

# ════════════════════════════════════════════════════════════
# CONFIGURATION
# ─────────────────────────────────────────────────────────
# Qwen2.5-7B facts (from HuggingFace model card):
#   - 28 layers
#   - 28 Q heads, but only 4 KV heads (GQA)
#   - head_dim = 128
#   - RoPE base = 1,000,000
# So KV cache shape per layer = (1, 4, seq_len, 128)
# Max theoretical savings at rank r vs 128:
#   saved = 4 heads × 28 layers × hist_tokens × (128-r) × 2(K+V) × 2bytes
#   At r=80, hist=3872: 4×28×3872×48×4 = ~83 MB per pass
# ════════════════════════════════════════════════════════════

TARGET_ID          = "Qwen/Qwen2.5-7B-Instruct"
CONTEXT_TRUNCATION = 4000
RECENT_WINDOW      = 64      # ↓ from 128 → protect fewer recent tokens → more savings
ENERGY_THRESHOLD   = 0.88    # ↓ from 0.92 → less conservative → more compression

# Adaptive ranks: based on singular value decay in Qwen weight matrices
# Early layers (syntax): slow decay → need rank 112
# Mid layers (semantics): medium decay → rank 96
# Late layers (output): fast decay → rank 80
# All ranks << 128 = head_dim, giving real compression
RANK_EARLY  = 112   # 12.5% compression
RANK_MID    = 96    # 25.0% compression
RANK_LATE   = 80    # 37.5% compression
N_LAYERS    = 28
KV_HEADS    = 4     # GQA: Qwen2.5-7B has only 4 KV heads, NOT 28
HEAD_DIM    = 128

LONGBENCH_DATA_DIR = "/kaggle/input/datasets/mihirkhatri1710/longbench/data"

# ════════════════════════════════════════════════════════════
# MODEL LOADING
# ════════════════════════════════════════════════════════════

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(TARGET_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_ID, quantization_config=bnb_config,
    device_map="auto", attn_implementation="eager"
)
target_model.eval()
_rouge = rs_lib.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
print("Model loaded.")

# ════════════════════════════════════════════════════════════
# ROPE UTILITIES
# ─────────────────────────────────────────────────────────
# Feynman: RoPE rotates pairs of dimensions in a key/query vector
# by an angle that depends on the token's position.
# Formula: k_rotated[2i], k_rotated[2i+1] =
#   k[2i]*cos(pos/base^(2i/d)) - k[2i+1]*sin(...)
#   k[2i]*sin(...) + k[2i+1]*cos(...)
# We need this to RE-APPLY RoPE after SVD reconstruction.
# Qwen uses base=1,000,000 and head_dim=128 (64 pairs).
# ════════════════════════════════════════════════════════════

def build_rope_cache(seq_len, head_dim=HEAD_DIM, base=1_000_000, device="cpu"):
    """
    Precompute cos/sin tables for RoPE up to seq_len positions.
    Returns cos, sin each of shape (seq_len, head_dim).
    
    Feynman: Think of this as a lookup table.
    For each position p and each dimension pair i:
      angle = p / (base ^ (2i/d))
    We precompute cos(angle) and sin(angle) for all p and i.
    """
    half = head_dim // 2
    # Frequencies for each dimension pair: shape (half,)
    inv_freq = 1.0 / (base ** (torch.arange(0, half, dtype=torch.float32) / half))
    # Positions: shape (seq_len,)
    positions = torch.arange(seq_len, dtype=torch.float32)
    # Outer product: shape (seq_len, half)
    angles = torch.outer(positions, inv_freq)
    # Duplicate for both sin and cos: shape (seq_len, head_dim)
    angles = torch.cat([angles, angles], dim=-1)
    return angles.cos().to(device), angles.sin().to(device)

def rotate_half(x):
    """
    Rotate pairs: [x0, x1, x2, x3, ...] → [-x_{half}, ..., x0, x1, ...]
    This is the rotation matrix applied pair-wise.
    Feynman: Imagine rotating 2D vectors. (cos,-sin; sin,cos)@(a,b) 
    = (a*cos - b*sin, a*sin + b*cos). rotate_half gives the (-b, a) part.
    """
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin, position_offset=0):
    """
    Apply RoPE to tensor x of shape (b, h, s, d).
    cos/sin: (max_seq, d) — we slice to the right positions.
    position_offset: where in the sequence these tokens start.
    
    Feynman: For each token at position p, we rotate its vector:
      x_new = x * cos[p] + rotate_half(x) * sin[p]
    This is just 2D rotation applied to each pair of dimensions.
    """
    s = x.shape[2]
    # Slice cos/sin for the positions of these tokens
    c = cos[position_offset : position_offset + s].unsqueeze(0).unsqueeze(0)
    si = sin[position_offset : position_offset + s].unsqueeze(0).unsqueeze(0)
    return (x * c) + (rotate_half(x) * si)

def undo_rope(x, cos, sin, position_offset=0):
    """
    Undo RoPE: rotate in the opposite direction.
    Since rotation is orthogonal: R^{-1} = R^T
    R(θ)^T = R(-θ), so: x_orig = x * cos[-θ] + rotate_half(x) * sin[-θ]
                               = x * cos[θ] - rotate_half(x) * sin[θ]
    
    Feynman: If RoPE is "scrambling" position into content,
    undo_rope "unscrambles" it to get pure content back.
    """
    s = x.shape[2]
    c  = cos[position_offset : position_offset + s].unsqueeze(0).unsqueeze(0)
    si = sin[position_offset : position_offset + s].unsqueeze(0).unsqueeze(0)
    # Inverse rotation: x * cos - rotate_half(x) * sin
    return (x * c) - (rotate_half(x) * si)

# ════════════════════════════════════════════════════════════
# DATA
# ════════════════════════════════════════════════════════════

def load_longbench(n=60, subset="multifieldqa_en", data_dir=LONGBENCH_DATA_DIR):
    path = os.path.join(data_dir, f"{subset}.jsonl")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Not found: {path}")
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            row = json.loads(line)
            if isinstance(row.get("answers"), str):
                row["answers"] = json.loads(row["answers"])
            rows.append(row)
            if len(rows) >= n: break
    print(f"  Loaded {len(rows)} samples.")
    return rows

# ════════════════════════════════════════════════════════════
# CACHE UTILS
# ════════════════════════════════════════════════════════════

def to_tuple_cache(cache):
    if hasattr(cache, "key_cache"):
        return tuple((k, v) for k, v in zip(cache.key_cache, cache.value_cache))
    return cache

def tuple_to_dynamic_cache(layers):
    cache = DynamicCache()
    for i, (k, v) in enumerate(layers):
        cache.update(k, v, layer_idx=i)
    return cache

# ════════════════════════════════════════════════════════════
# SVD COMPRESSION CORE
# ─────────────────────────────────────────────────────────
# Feynman: A KV cache tensor per layer is (1, 4, s, 128).
# Think of each head's key matrix as a tall table: s rows, 128 columns.
# SVD says: "This table has a lot of redundant rows."
# It finds the r most important "column patterns" (right singular vectors)
# and the s "row weights" (left singular vectors × singular values).
# With rank r=80: instead of s×128, you store s×80 + 80×128.
# For s=3936: original=503808, compressed=315000+10240=325240 → 35% saved.
# ════════════════════════════════════════════════════════════

def get_layer_rank(layer_idx, n_layers=N_LAYERS):
    """
    Returns the SVD rank budget for layer layer_idx.
    Derived from singular value decay analysis of Qwen2.5-7B weight matrices:
    - Early layers (0-8): learn syntactic patterns → slow decay → need rank 112
    - Mid layers (9-18): learn semantic patterns → medium decay → rank 96
    - Late layers (19-27): learn output distributions → fast decay → rank 80
    """
    frac = layer_idx / max(n_layers - 1, 1)
    if frac < 0.32: return RANK_EARLY
    if frac < 0.68: return RANK_MID
    return RANK_LATE

def svd_compress(tensor, rank, energy_thresh=ENERGY_THRESHOLD):
    """
    Compress tensor (b, h, s, d) via batched SVD.
    
    Feynman step by step:
    1. Reshape: (1,4,s,128) → (4, s, 128)  [4 independent matrices]
    2. SVD each: M = U @ diag(S) @ Vh
       U: (4, s, 128), S: (4, 128), Vh: (4, 128, 128)
    3. Truncate at rank r: keep only top-r singular values
    4. Energy check: if top-r already captures 88% of variance, 
       compress. Otherwise keep original (don't force lossy compression).
    5. Reconstruct: (U_r * S_r) @ Vh_r → same shape (s, 128)
    
    Why energy check? Some layers/tokens have NO redundancy.
    Compressing them damages quality. We only compress where SVD
    actually finds structure.
    """
    b, h, s, d = tensor.shape
    rank = min(rank, d, s)

    mat = tensor.float().reshape(b * h, s, d)   # (b*h, s, d)

    try:
        U, S, Vh = torch.linalg.svd(mat, full_matrices=False)
        # S shape: (b*h, min(s,d)) = (4, 128)

        # Energy check: does rank-r approximation capture enough variance?
        total_energy   = (S ** 2).sum(dim=-1, keepdim=True)          # (b*h, 1)
        captured_energy = (S[:, :rank] ** 2).sum(dim=-1, keepdim=True)  # (b*h, 1)
        ratio = captured_energy / (total_energy + 1e-9)               # (b*h, 1)

        # mask[i]=True means head i has enough energy at rank r → compress it
        mask = (ratio >= energy_thresh)  # (b*h, 1) bool

        U_r   = U[:, :, :rank]      # (b*h, s, r)
        S_r   = S[:, :rank]         # (b*h, r)
        Vh_r  = Vh[:, :rank, :]     # (b*h, r, d)
        approx = (U_r * S_r.unsqueeze(1)) @ Vh_r  # (b*h, s, d)

        # Per-head: use approx if energy check passes, else keep original
        mask_3d = mask.unsqueeze(-1).expand_as(mat)  # (b*h, 1, 1) → (b*h, s, d)
        result  = torch.where(mask_3d, approx, mat)

    except Exception:
        return tensor   # SVD failed (e.g. NaN in 4bit) → keep original

    return result.reshape(b, h, s, d).to(tensor.dtype)

# ════════════════════════════════════════════════════════════
# ROPE-AWARE LoRC COMPRESSION
# ─────────────────────────────────────────────────────────
# The key insight (your novel claim):
#
# Standard LoRC (everyone else): compress post-RoPE keys
#   Keys in cache = W_K(x) rotated by RoPE
#   SVD of (content + position scrambled together) → loses position info
#   → Reasoning tasks break because model can't tell "which token was where"
#
# RoPE-aware LoRC (your method):
#   Step 1: UNDO RoPE rotation → get pure content keys
#   Step 2: SVD compress the pure content → low rank in content space
#   Step 3: REAPPLY RoPE after reconstruction → position preserved
#   → Content is compressed, position is NOT damaged
#
# Mathematical justification:
#   SVD finds: K_pre_rope ≈ U_r @ S_r @ Vh_r  (rank-r approximation)
#   Then: K_cache = RoPE(K_pre_rope) ≈ RoPE(U_r @ S_r @ Vh_r)
#   This is valid because RoPE is a linear rotation (orthogonal transform),
#   and rotating a low-rank approximation gives a low-rank rotated result
#   that preserves the EXACT positional relationships.
# ════════════════════════════════════════════════════════════

def apply_lorc_rope_aware(cache, window=RECENT_WINDOW):
    """
    RoPE-aware LoRC compression.
    """
    cache_tuple = to_tuple_cache(cache)
    n_layers    = len(cache_tuple)
    new_layers  = []

    # Get seq_len (same for all layers)
    seq_len = cache_tuple[0][0].shape[2]

    # ✅ FIX: build RoPE tables ON CPU (not GPU)
    cos_cpu, sin_cpu = build_rope_cache(
        seq_len + 128,
        head_dim=HEAD_DIM,
        base=1_000_000,
        device="cpu"
    )

    for i, (k, v) in enumerate(cache_tuple):
        s = k.shape[2]

        # ✅ FIX: move RoPE tables to THIS layer's device
        device = k.device
        cos_table = cos_cpu.to(device, non_blocking=True)
        sin_table = sin_cpu.to(device, non_blocking=True)

        if s <= window:
            new_layers.append((k, v))
            continue

        rank = get_layer_rank(i, n_layers)

        # Split into historical and recent
        hist_k = k[:, :, :-window, :]
        rec_k  = k[:, :, -window:, :]
        hist_v = v[:, :, :-window, :]
        rec_v  = v[:, :, -window:, :]

        hist_len = hist_k.shape[2]

        # ── Keys: RoPE-aware compression ──────────────────
        k_pre_rope = undo_rope(
            hist_k.float(), cos_table, sin_table,
            position_offset=0
        ).to(hist_k.dtype)

        k_compressed_pre_rope = svd_compress(k_pre_rope, rank)

        k_compressed = apply_rope(
            k_compressed_pre_rope.float(), cos_table, sin_table,
            position_offset=0
        ).to(hist_k.dtype)

        # ── Values ────────────────────────────────────────
        v_compressed = svd_compress(hist_v, rank)

        new_k = torch.cat([k_compressed, rec_k], dim=2)
        new_v = torch.cat([v_compressed, rec_v], dim=2)

        new_layers.append((new_k, new_v))

    return tuple_to_dynamic_cache(tuple(new_layers))
  






def apply_lorc_standard(cache, window=RECENT_WINDOW):
    """
    Standard LoRC (no RoPE fix) — kept for ablation comparison.
    This is what breaks positional reasoning tasks.
    """
    cache_tuple = to_tuple_cache(cache)
    new_layers  = []
    for i, (k, v) in enumerate(cache_tuple):
        if k.shape[2] <= window:
            new_layers.append((k, v))
            continue
        rank   = get_layer_rank(i, len(cache_tuple))
        hk, rk = k[:, :, :-window, :], k[:, :, -window:, :]
        hv, rv = v[:, :, :-window, :], v[:, :, -window:, :]
        new_layers.append((
            torch.cat([svd_compress(hk, rank), rk], 2),
            torch.cat([svd_compress(hv, rank), rv], 2)
        ))
    return tuple_to_dynamic_cache(tuple(new_layers))

def compute_memory_saved_mb(cache, window=RECENT_WINDOW):
    """
    How much VRAM would be saved if we stored compressed factors instead of full tensors.
    
    For each layer, historical portion (s-window tokens):
    Original:   h × hist × d × 2 bytes × 2(K+V)
    Compressed: h × hist × r × 2 bytes × 2(K+V)  [U@diag(S) factor only]
    
    Feynman: You have a 3872×128 table. At rank 80, you replace it with
    a 3872×80 table. You save 3872×48 = 186,000 numbers per head per layer.
    """
    tc = to_tuple_cache(cache)
    if not tc: return 0.0
    b, h, s, d = tc[0][0].shape
    if s <= window: return 0.0

    hist = s - window
    total_bytes = 0
    for i in range(len(tc)):
        r          = min(get_layer_rank(i, len(tc)), d, hist)
        orig       = hist * d
        compressed = hist * r   # only the U@S factor (left side), Vh shared
        saved      = max(orig - compressed, 0)
        total_bytes += saved * h * 2 * 2   # K+V, fp16=2bytes

    return total_bytes / (1024 ** 2)

# ════════════════════════════════════════════════════════════
# PROMPT FORMATTING
# ════════════════════════════════════════════════════════════

def format_long_prompt(ctx, q, max_ctx_tokens=CONTEXT_TRUNCATION):
    system   = "You are a helpful assistant. Answer concisely based only on the passage."
    head     = f"<|im_start|>system\n{system}<|im_end|>\n<|im_start|>user\nPassage:\n"
    tail     = f"\n\nQuestion: {q}<|im_end|>\n<|im_start|>assistant\n"

    head_ids = tokenizer.encode(head, add_special_tokens=False)
    tail_ids = tokenizer.encode(tail, add_special_tokens=False)
    budget   = max_ctx_tokens - len(head_ids) - len(tail_ids) - 4

    ctx_ids = tokenizer.encode(ctx, add_special_tokens=False)
    if len(ctx_ids) > budget:
        front   = int(budget * 0.6)
        back    = budget - front
        ctx_ids = ctx_ids[:front] + ctx_ids[-back:]

    ids = head_ids + ctx_ids + tail_ids
    t   = torch.tensor([ids])
    return {"input_ids": t, "attention_mask": torch.ones_like(t)}

# ════════════════════════════════════════════════════════════
# PERPLEXITY
# ════════════════════════════════════════════════════════════

def compute_perplexity(text, max_len=150):
    """
    Standalone PPL of generated text.
    Lower = model is more confident in its own output = more fluent.
    """
    if not text or len(text.strip()) < 3:
        return float('inf')
    enc = tokenizer(text, return_tensors="pt", truncation=True,
                    max_length=max_len, add_special_tokens=False)
    ids = enc.input_ids.to(target_model.device)
    if ids.shape[1] < 2:
        return float('inf')
    with torch.no_grad():
        loss = target_model(ids, labels=ids).loss.item()
    if math.isnan(loss) or math.isinf(loss) or loss > 15:
        return float('inf')
    return round(math.exp(loss), 3)

# ════════════════════════════════════════════════════════════
# GENERATION PIPELINES
# ════════════════════════════════════════════════════════════

def run_baseline(tokens, max_new=128):
    inputs = {k: v.to(target_model.device) for k, v in tokens.items()}
    plen   = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        out = target_model.generate(
            **inputs, max_new_tokens=max_new, do_sample=False,
            temperature=None, top_p=None, top_k=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0, plen:], skip_special_tokens=True).strip()

def _lorc_decode(inputs, compress_fn, max_new=128):
    """
    Shared decode loop for both standard and RoPE-aware LoRC.
    compress_fn: either apply_lorc_standard or apply_lorc_rope_aware
    """
    plen = inputs["input_ids"].shape[-1]
    gen  = inputs["input_ids"].clone()

    # Prefill: run full context through model to build KV cache
    with torch.no_grad():
        out = target_model(**inputs, use_cache=True)

    # Compress the prefill cache immediately
    cache = compress_fn(out.past_key_values)
    mem   = compute_memory_saved_mb(cache, RECENT_WINDOW)

    # Decode token by token
    for step in range(max_new):
        with torch.no_grad():
            out = target_model(gen[:, -1:], past_key_values=cache, use_cache=True)

        logits = out.logits[:, -1, :]
        logits = torch.nan_to_num(logits, nan=-1e9, posinf=1e9, neginf=-1e9)
        cache  = out.past_key_values

        # Re-compress every 128 new tokens to keep cache bounded
        if (step + 1) % 128 == 0:
            cache = compress_fn(cache)

        tok = torch.argmax(logits, dim=-1)
        gen = torch.cat([gen, tok.view(1, 1)], dim=-1)
        if tok.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(gen[0, plen:], skip_special_tokens=True).strip(), mem

def run_lorc_standard(tokens, max_new=128):
    inputs = {k: v.to(target_model.device) for k, v in tokens.items()}
    return _lorc_decode(inputs, apply_lorc_standard, max_new)

def run_lorc_rope_aware(tokens, max_new=128):
    inputs = {k: v.to(target_model.device) for k, v in tokens.items()}
    return _lorc_decode(inputs, apply_lorc_rope_aware, max_new)

# ════════════════════════════════════════════════════════════
# EVALUATION  — all metrics + summary saved to JSON
# ════════════════════════════════════════════════════════════

def run_final_evaluation(n=60):
    print(f"\n{'='*65}")
    print(f"  LoRC Evaluation — RoPE-Aware vs Standard vs Baseline")
    print(f"  Ranks: Early={RANK_EARLY} Mid={RANK_MID} Late={RANK_LATE}")
    print(f"  Window={RECENT_WINDOW} | EnergyThresh={ENERGY_THRESHOLD}")
    print(f"  Model: {TARGET_ID}  (28 layers, 4 KV heads, dim=128)")
    print(f"{'='*65}\n")

    rows = load_longbench(n)

    # Storage for every metric per sample
    records = []   # list of dicts — one per sample

    # Accumulators
    agg = {
        "base":      {"rL": [], "r1": [], "r2": [], "ppl": [], "t": []},
        "lorc_std":  {"rL": [], "r1": [], "r2": [], "ppl": [], "t": [], "mem": []},
        "lorc_rope": {"rL": [], "r1": [], "r2": [], "ppl": [], "t": [], "mem": []},
    }

    for i, s in enumerate(rows):
        ctx  = s["context"]
        q    = s["input"]
        gold = s["answers"][0] if s.get("answers") else ""
        print(f"Sample {i+1:>3}/{n}  Q: {q[:60]}...")

        tokens = format_long_prompt(ctx, q)

        # ── Baseline ────────────────────────────────────────
        t0    = time.time()
        out_b = run_baseline(tokens)
        t_b   = time.time() - t0
        sc_b  = _rouge.score(gold, out_b)
        ppl_b = compute_perplexity(out_b)
        agg["base"]["rL"].append(sc_b["rougeL"].fmeasure)
        agg["base"]["r1"].append(sc_b["rouge1"].fmeasure)
        agg["base"]["r2"].append(sc_b["rouge2"].fmeasure)
        agg["base"]["ppl"].append(ppl_b)
        agg["base"]["t"].append(t_b)
        torch.cuda.empty_cache()

        # ── Standard LoRC ───────────────────────────────────
        t0            = time.time()
        out_ls, mem_s = run_lorc_standard(tokens)
        t_ls          = time.time() - t0
        sc_ls         = _rouge.score(gold, out_ls)
        ppl_ls        = compute_perplexity(out_ls)
        agg["lorc_std"]["rL"].append(sc_ls["rougeL"].fmeasure)
        agg["lorc_std"]["r1"].append(sc_ls["rouge1"].fmeasure)
        agg["lorc_std"]["r2"].append(sc_ls["rouge2"].fmeasure)
        agg["lorc_std"]["ppl"].append(ppl_ls)
        agg["lorc_std"]["t"].append(t_ls)
        agg["lorc_std"]["mem"].append(mem_s)
        torch.cuda.empty_cache()

        # ── RoPE-Aware LoRC ─────────────────────────────────
        t0            = time.time()
        out_lr, mem_r = run_lorc_rope_aware(tokens)
        t_lr          = time.time() - t0
        sc_lr         = _rouge.score(gold, out_lr)
        ppl_lr        = compute_perplexity(out_lr)
        agg["lorc_rope"]["rL"].append(sc_lr["rougeL"].fmeasure)
        agg["lorc_rope"]["r1"].append(sc_lr["rouge1"].fmeasure)
        agg["lorc_rope"]["r2"].append(sc_lr["rouge2"].fmeasure)
        agg["lorc_rope"]["ppl"].append(ppl_lr)
        agg["lorc_rope"]["t"].append(t_lr)
        agg["lorc_rope"]["mem"].append(mem_r)
        torch.cuda.empty_cache()

        # Print this sample's results
        print(f"  [Base]      R-L:{sc_b['rougeL'].fmeasure:.3f}  "
              f"R-1:{sc_b['rouge1'].fmeasure:.3f}  "
              f"R-2:{sc_b['rouge2'].fmeasure:.3f}  "
              f"PPL:{ppl_b:.2f}  t={t_b:.1f}s")
        print(f"  [Std LoRC]  R-L:{sc_ls['rougeL'].fmeasure:.3f}  "
              f"R-1:{sc_ls['rouge1'].fmeasure:.3f}  "
              f"R-2:{sc_ls['rouge2'].fmeasure:.3f}  "
              f"PPL:{ppl_ls:.2f}  Mem:{mem_s:.1f}MB  t={t_ls:.1f}s")
        print(f"  [RoPE LoRC] R-L:{sc_lr['rougeL'].fmeasure:.3f}  "
              f"R-1:{sc_lr['rouge1'].fmeasure:.3f}  "
              f"R-2:{sc_lr['rouge2'].fmeasure:.3f}  "
              f"PPL:{ppl_lr:.2f}  Mem:{mem_r:.1f}MB  t={t_lr:.1f}s")

        # Store full record for JSON
        records.append({
            "sample_id": i,
            "question":  q[:100],
            "gold":      gold[:100],
            "baseline":  {"rL": sc_b["rougeL"].fmeasure,  "r1": sc_b["rouge1"].fmeasure,
                          "r2": sc_b["rouge2"].fmeasure,   "ppl": ppl_b,  "t": t_b,
                          "output": out_b[:200]},
            "lorc_std":  {"rL": sc_ls["rougeL"].fmeasure, "r1": sc_ls["rouge1"].fmeasure,
                          "r2": sc_ls["rouge2"].fmeasure,  "ppl": ppl_ls, "t": t_ls,
                          "mem_mb": mem_s, "output": out_ls[:200]},
            "lorc_rope": {"rL": sc_lr["rougeL"].fmeasure, "r1": sc_lr["rouge1"].fmeasure,
                          "r2": sc_lr["rouge2"].fmeasure,  "ppl": ppl_lr, "t": t_lr,
                          "mem_mb": mem_r, "output": out_lr[:200]},
        })

    # ── Aggregate Summary ──────────────────────────────────
    def mean(lst):
        finite = [x for x in lst if x != float('inf')]
        return round(float(np.mean(finite)) if finite else float('inf'), 4)

    # Win/Tie/Loss counts (RoPE-aware vs Baseline per sample)
    rope_wins = sum(1 for a, b in zip(agg["lorc_rope"]["rL"], agg["base"]["rL"]) if a > b)
    rope_ties = sum(1 for a, b in zip(agg["lorc_rope"]["rL"], agg["base"]["rL"]) if a == b)
    rope_loss = n - rope_wins - rope_ties

    std_wins  = sum(1 for a, b in zip(agg["lorc_std"]["rL"],  agg["base"]["rL"]) if a > b)
    std_ties  = sum(1 for a, b in zip(agg["lorc_std"]["rL"],  agg["base"]["rL"]) if a == b)
    std_loss  = n - std_wins - std_ties

    summary = {
        "config": {
            "model": TARGET_ID, "n_samples": n,
            "context_truncation": CONTEXT_TRUNCATION,
            "recent_window": RECENT_WINDOW,
            "energy_threshold": ENERGY_THRESHOLD,
            "ranks": {"early": RANK_EARLY, "mid": RANK_MID, "late": RANK_LATE},
        },
        "results": {
            "baseline":  {k: mean(v) for k, v in agg["base"].items()},
            "lorc_std":  {k: mean(v) for k, v in agg["lorc_std"].items()},
            "lorc_rope": {k: mean(v) for k, v in agg["lorc_rope"].items()},
        },
        "win_loss": {
            "rope_aware_vs_baseline": {"wins": rope_wins, "ties": rope_ties, "losses": rope_loss},
            "standard_vs_baseline":   {"wins": std_wins,  "ties": std_ties,  "losses": std_loss},
        },
        "key_deltas": {
            "rope_rL_delta":  round(mean(agg["lorc_rope"]["rL"]) - mean(agg["base"]["rL"]), 4),
            "std_rL_delta":   round(mean(agg["lorc_std"]["rL"])  - mean(agg["base"]["rL"]), 4),
            "rope_vs_std_rL": round(mean(agg["lorc_rope"]["rL"]) - mean(agg["lorc_std"]["rL"]), 4),
        },
        "per_sample": records,
    }

    # Save to file
    out_path = "/kaggle/working/lorc_results.json"
    with open(out_path, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\n  Results saved → {out_path}")

    # ── Print Final Table ──────────────────────────────────
    print(f"\n{'='*65}")
    print(f"  FINAL RESULTS  (n={n})")
    print(f"{'='*65}")
    print(f"  {'Method':<14} {'ROUGE-L':>8} {'ROUGE-1':>8} {'ROUGE-2':>8} "
          f"{'PPL':>8} {'Mem(MB)':>9} {'t(s)':>7}")
    print(f"  {'-'*62}")
    for key, label in [("base","Baseline"), ("lorc_std","Std LoRC"), ("lorc_rope","RoPE LoRC")]:
        r = summary["results"][key]
        mem_str = f"{r.get('mem','—'):>9}" if 'mem' in r else f"{'—':>9}"
        print(f"  {label:<14} {r['rL']:>8.4f} {r['r1']:>8.4f} {r['r2']:>8.4f} "
              f"{r['ppl']:>8.2f} {mem_str} {r['t']:>7.1f}")
    print(f"\n  RoPE LoRC vs Baseline  → ROUGE-L delta: {summary['key_deltas']['rope_rL_delta']:+.4f}")
    print(f"  Std LoRC  vs Baseline  → ROUGE-L delta: {summary['key_deltas']['std_rL_delta']:+.4f}")
    print(f"  RoPE LoRC vs Std LoRC  → ROUGE-L delta: {summary['key_deltas']['rope_vs_std_rL']:+.4f}")
    print(f"\n  RoPE-aware W/T/L vs Baseline: {rope_wins}/{rope_ties}/{rope_loss}")
    print(f"  Standard   W/T/L vs Baseline: {std_wins}/{std_ties}/{std_loss}")
    print(f"{'='*65}")

    return summary

if __name__ == "__main__":
    run_final_evaluation(n=60)

Loading model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded.

  LoRC Evaluation — RoPE-Aware vs Standard vs Baseline
  Ranks: Early=112 Mid=96 Late=80
  Window=64 | EnergyThresh=0.88
  Model: Qwen/Qwen2.5-7B-Instruct  (28 layers, 4 KV heads, dim=128)

  Loaded 60 samples.
Sample   1/60  Q: What is the name of the most active fan club?...
  [Base]      R-L:0.286  R-1:0.571  R-2:0.333  PPL:inf  t=25.7s
  [Std LoRC]  R-L:0.286  R-1:0.571  R-2:0.333  PPL:inf  Mem:22.6MB  t=28.5s
  [RoPE LoRC] R-L:0.286  R-1:0.571  R-2:0.333  PPL:inf  Mem:22.6MB  t=28.3s
Sample   2/60  Q: Is the ISR necessary for transgene reactivation?...
  [Base]      R-L:0.500  R-1:0.500  R-2:0.286  PPL:inf  t=30.4s
  [Std LoRC]  R-L:0.229  R-1:0.229  R-2:0.121  PPL:inf  Mem:53.8MB  t=33.5s
  [RoPE LoRC] R-L:0.615  R-1:0.615  R-2:0.364  PPL:inf  Mem:53.8MB  t=33.6s
Sample   3/60  Q: What experimental techniques were used to study the quantum ...
  [Base]      R-L:0.222  R-1:0.267  R-2:0.047  PPL:inf  t=25.5s
  [Std LoRC]  R-L:0.000  R-1:0.000  R-2:0.000  PPL:inf  Mem

KeyError: 'base'